In [21]:
import pandas as pd
import shapefile
import chardet
pd.options.mode.chained_assignment = None
from pyproj import CRS
import pyproj
from tqdm import tqdm
import os
import math
from osgeo import gdal
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
gdal.UseExceptions()

## 3d

In [29]:
slope_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\slope\鬼怒川05m_03_slope.asc"
aspect_diff_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\aspect\dif\鬼怒川05m_03_aspect_diff.asc"
maxc_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\maxcurv\鬼怒川05m_03_maxcurv.asc"
re_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\relativeele\鬼怒川05m_03_relative_ele.asc"
levee_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\levee\1\鬼怒川05m_03_levee.asc"
elevation_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\asc\鬼怒川05m_03.asc"
extent, _, _, cellsize, aspect = ras_info(aspect_diff_path)
_, _, _, _, maxcurv = ras_info(maxc_path)
_, _, _, _, relative_ele = ras_info(re_path)
_, _, _, _, slope = ras_info(slope_path)
_, _, _, _, elevation = ras_info(elevation_path)

In [30]:
def mask_com(data, nodata_value):
    
    data = np.pad(data, pad_width=2, mode='edge')
    windows = sliding_window_view(data, (5, 5))

    compute_mask = np.all(windows != nodata_value, axis=(2, 3))  
    return windows, compute_mask

In [59]:
windows, compute_mask = mask_com(elevation, -9999)
windows_max, compute_mask = mask_com(maxcurv, -9999)

In [13]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from ipywidgets import interact, IntSlider
import numpy as np
from matplotlib import cm

gyou, retu = 1241, 1697
# 座標と標高・曲率
x = y = np.arange(5)*5 
X, Y = np.meshgrid(x, y)
Z = X ** 2 - Y ** 2#windows[gyou][retu]
laplacian = (
    -4 * Z + np.roll(Z, 1, axis=0) + np.roll(Z, -1, axis=0)
    + np.roll(Z, 1, axis=1) + np.roll(Z, -1, axis=1)
)#windows_max[gyou][retu]

# 地面とする高さ
ground = Z.min()

@interact(elev=IntSlider(min=-180, max=180, step=5, value=45, continuous_update=False),
          azim=IntSlider(min=-180, max=180, step=5, value=90, continuous_update=False))
def plot_3d(elev, azim):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    # 側面ポリゴンを生成（4辺）
    verts = []

    rows, cols = Z.shape

    # 各辺のセルを底面と結ぶ（上下左右）
    for i in range(rows - 1):
        # 左側
        verts.append([
            (X[i, 0],     Y[i, 0],     ground),
            (X[i, 0],     Y[i, 0],     Z[i, 0]),
            (X[i+1, 0],   Y[i+1, 0],   Z[i+1, 0]),
            (X[i+1, 0],   Y[i+1, 0],   ground),
        ])
        # 右側
        verts.append([
            (X[i, -1],     Y[i, -1],     ground),
            (X[i, -1],     Y[i, -1],     Z[i, -1]),
            (X[i+1, -1],   Y[i+1, -1],   Z[i+1, -1]),
            (X[i+1, -1],   Y[i+1, -1],   ground),
        ])
    for j in range(cols - 1):
        # 下側
        verts.append([
            (X[0, j],     Y[0, j],     ground),
            (X[0, j],     Y[0, j],     Z[0, j]),
            (X[0, j+1],   Y[0, j+1],   Z[0, j+1]),
            (X[0, j+1],   Y[0, j+1],   ground),
        ])
        # 上側
        verts.append([
            (X[-1, j],     Y[-1, j],     ground),
            (X[-1, j],     Y[-1, j],     Z[-1, j]),
            (X[-1, j+1],   Y[-1, j+1],   Z[-1, j+1]),
            (X[-1, j+1],   Y[-1, j+1],   ground),
        ])

    # 底面ポリゴン（外周を一筆書き）
    bottom_ring = []
    # 下辺 → 右辺 → 上辺 → 左辺（時計回り）
    for j in range(cols):
        bottom_ring.append((X[0, j], Y[0, j], ground))
    for i in range(1, rows):
        bottom_ring.append((X[i, -1], Y[i, -1], ground))
    for j in range(cols-2, -1, -1):
        bottom_ring.append((X[-1, j], Y[-1, j], ground))
    for i in range(rows-2, 0, -1):
        bottom_ring.append((X[i, 0], Y[i, 0], ground))
    verts.append(bottom_ring)

    # 描画：側面と底面
    poly = Poly3DCollection(verts, facecolor='green', edgecolor='black', alpha=1)
    ax.add_collection3d(poly)
    # 地形のカラーマップ（曲率を正規化）
    norm_lap = (laplacian - laplacian.min()) / (laplacian.max() - laplacian.min())
    facecolors = cm.coolwarm(norm_lap)

    # 地形上面を描画
    ax.plot_surface(X, Y, Z, facecolors=facecolors, rstride=1, cstride=1, linewidth=0, antialiased=False)


    # 視点
    ax.view_init(elev=elev, azim=azim)
    plt.show()


interactive(children=(IntSlider(value=45, continuous_update=False, description='elev', max=180, min=-180, step…

## 前処理(bilinear)

In [9]:
#ras_info
def ras_info(path):
    src = gdal.Open(path, gdal.GA_ReadOnly)
    left, xleng, rotx, top, roty, yleng = src.GetGeoTransform()
    
    xsize, ysize, cellsize = src.RasterXSize, src.RasterYSize, xleng
    
    botm = top + ysize * yleng
    right = left + xsize * xleng
    extent = [left, botm, right, top]
    
    data = src.GetRasterBand(1).ReadAsArray()
    return extent, xsize, ysize, cellsize, data

def save_asc(raster, extent, resolution, asc_path):

    rows, cols = raster.shape
    x_min, y_min = extent[0], extent[1]
    header = f"NCOLS {cols}\nNROWS {rows}\nXLLCORNER {x_min}\nYLLCORNER {y_min}\n"
    header += f"CELLSIZE {resolution}\nNODATA_VALUE -9999\n"

    with open(asc_path, "w") as f:
        f.write(header)
        np.savetxt(f, raster, delimiter=" ", fmt="%.4f")
        
def bilinear_resample_from_file(raster_path, extent_out, reso_out, asc_path, nodata_value=-9999):
    
    extent_in, xsize, ysize, cellsize_in, data_in = ras_info(raster_path)
    
    left_in, bottom_in, right_in, top_in = extent_in
    left_out, bottom_out, right_out, top_out = extent_out
    
    x_coords = np.arange(left_out + 0.5 * reso_out, right_out, reso_out)
    y_coords = np.arange(top_out - 0.5 * reso_out, bottom_out, - reso_out)
    
    # 出力グリッド
    x_grid, y_grid = np.meshgrid(x_coords, y_coords)

    #出力グリッドを入力グリッドに投影
    x_pos = (x_grid - left_in) / cellsize_in
    y_pos = (top_in - y_grid) / cellsize_in
    
    x0, y0, x1, y1 = (np.floor(pos).astype(int) for pos in [x_pos, y_pos, x_pos + 1, y_pos + 1 ])
    dx, dy = x_pos - x0, y_pos - y0

    data_out = np.full(x_grid.shape, nodata_value, dtype=np.float32)

    # 有効な範囲のマスク（全ピクセル）
    mask = (x0 >= 0) & (x1 < xsize) & (y0 >= 0) & (y1 < ysize)

    if np.any(mask):
        # 安全に flat index にして抽出
        idx = np.where(mask)
        x0v, x1v, y0v, y1v, dxv, dyv = (arr[idx] for arr in [x0, x1, y0, y1, dx, dy])
        
    coords = [(y0v, x0v), (y0v, x1v), (y1v, x0v), (y1v, x1v)]
    Q11, Q21, Q12, Q22 = (data_in[y, x] for y, x in coords)

    # NoDataマスク（各画素ごとに）
    m11, m21, m12, m22 = (arr != nodata_value for arr in [Q11, Q21, Q12, Q22])

    w11 = (1 - dxv) * (1 - dyv) * m11
    w21 = dxv * (1 - dyv) * m21
    w12 = (1 - dxv) * dyv * m12
    w22 = dxv * dyv * m22
    
    weight_sum = w11 + w21 + w12 + w22
    
    value_sum = (
        np.where(m11, Q11 * w11, 0) +
        np.where(m21, Q21 * w21, 0) +
        np.where(m12, Q12 * w12, 0) +
        np.where(m22, Q22 * w22, 0)
    )
    out_values = np.where(weight_sum > 0, value_sum / weight_sum, nodata_value)

    # 出力に代入
    data_out[idx] = out_values
    save_asc(data_out, extent_out, reso_out, asc_path)
    
#使用例

#extent1 = ocellsize * np.round(np.array(extent) / ocellsize)
#bilinear_resample_from_file(raster_path, extent1, ocellsize, asc_path, nodata_value=-9999)

In [15]:
ocellsize = 5
raster_path = r"D:\安部川水系\tif\523843.tif"
asc_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\安部川水系\05m\elevation\安部川05m_04.asc"
extent, xsize, ysize, cellsize, data = ras_info(raster_path)
extent1 = ocellsize * np.round(np.array(extent) / ocellsize)
bilinear_resample_from_file(raster_path, extent1, ocellsize, asc_path, nodata_value=-9999)

C:\Users\pc\AppData\Local\Temp\ipykernel_860\4184354464.py:75: RuntimeWarning: invalid value encountered in divide
  out_values = np.where(weight_sum > 0, value_sum / weight_sum, nodata_value)


# all

In [9]:
def ras_info(path):
    src = gdal.Open(path, gdal.GA_ReadOnly)
    left, xleng, rotx, top, roty, yleng = src.GetGeoTransform()
    
    xsize, ysize, cellsize = src.RasterXSize, src.RasterYSize, xleng
    
    botm = top + ysize * yleng
    right = left + xsize * xleng
    extent = [left, botm, right, top]
    
    data = src.GetRasterBand(1).ReadAsArray()
    return extent, xsize, ysize, cellsize, data

def save_asc(raster, extent, resolution, asc_path):
    rows, cols = raster.shape
    
    x_min, y_min = extent[0], extent[1]
    header = f"NCOLS {cols}\nNROWS {rows}\nXLLCORNER {x_min}\nYLLCORNER {y_min}\n"
    header += f"CELLSIZE {resolution}\nNODATA_VALUE -9999\n"

    with open(asc_path, "w") as f:
        f.write(header)
        np.savetxt(f, raster, delimiter=" ", fmt="%.4f")

def mask_com(data, nodata_value):
    
    data = np.pad(data, pad_width=1, mode='edge')
    windows = sliding_window_view(data, (3, 3))

    compute_mask = np.all(windows != nodata_value, axis=(2, 3))  
    return windows, compute_mask

def gaussian_kernel_3x3(sigma=0.8493):

    ax = np.arange(-1, 2)
    xx, yy = np.meshgrid(ax, ax)
    kernel = 1/(2*np.pi**2)*np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel /= np.sum(kernel)
    return np.round(kernel, 4)

def gaussian_smoothing(raster_path, gaussian_path, nodata_value, sigma=0.8493):

    kernel = gaussian_kernel_3x3(sigma)

    extent, _, _, cellsize, data = ras_info(raster_path)
    win, compute_mask = mask_com(data, nodata_value)
    gaussian = np.sum(win * kernel, axis=(2, 3))
    gaussian = np.where(compute_mask, gaussian, -9999)
    save_asc(gaussian, extent, cellsize, gaussian_path)
    
def compute_slope_aspect(raster_path, slope_path, aspect_path, aspect_diff_path,nodata_value):

    # 勾配計算用カーネル
    extent, _, _, cellsize, data = ras_info(raster_path)
    win, compute_mask = mask_com(data, nodata_value)
    
    kx = np.array([[-1, 0, 1],[-2, 0, 2],[-1, 0, 1]]) / (8 * cellsize)
    ky = np.array([[-1, -2, -1],[ 0, 0, 0],[ 1, 2, 1]]) / (8 * cellsize)

    dz_dx, dz_dy = (np.sum(win * kernel, axis=(2, 3)) for kernel in [kx, ky])

    slope_deg, aspect_deg = (np.full_like(dz_dx, nodata_value, dtype=float) for _ in range(2))
    
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg[compute_mask] = np.degrees(slope_rad[compute_mask])
    
    aspect_rad = np.atan2(dz_dy, -dz_dx)
    aspect_deg[compute_mask] = (90 - np.degrees(aspect_rad[compute_mask])) % 360
    
    flat_mask = (dz_dx == 0) & (dz_dy == 0) & compute_mask
    aspect_deg[flat_mask] = -1
    
    #aspect_diff
    winas, _ = mask_com(aspect_deg, nodata_value)
    dir_pairs = [((1, 2), (1, 0)),  # E-W
                 ((2, 1), (0, 1)),  # S-N
                 ((0, 2), (2, 0)),  # NE-SW
                 ((2, 2), (0, 0))]  # SE-NW

    # ペア1とペア2をそれぞれ抽出してスタック
    pair1, pair2 = zip(*[
        (winas[:, :, y1, x1], winas[:, :, y2, x2])
        for (y1, x1), (y2, x2) in dir_pairs
    ])
    pair1, pair2 = (np.stack(pair, axis=0) for pair in [pair1, pair2])

    # 180度回転（opposite）と差分（0–180度）を計算
    pair1_opposite = (pair1 + 180) % 360
    diff = np.abs(pair1_opposite - pair2)% 360
    diff = np.minimum(diff, 360 - diff)
    invalid_mask = (pair1 == nodata_value) | (pair2 == nodata_value)
    diff = np.where(invalid_mask, np.nan, diff)
    min_diff = np.nanmin(diff, axis=0)
    mask = (compute_mask) | ((dz_dx != 0) | (dz_dy != 0))
    min_diff = np.where(mask, min_diff, nodata_value)
    
    save_asc(slope_deg, extent, cellsize, slope_path)
    save_asc(aspect_deg, extent, cellsize, aspect_path)
    save_asc(min_diff, extent, cellsize, aspect_diff_path)

# curvature

def compute_pqrst(raster_path, nodata_value):
    
    extent, _, _, cellsize, data = ras_info(raster_path)
    win, compute_mask = mask_com(data, nodata_value)
    #numpy
    kx = np.array([[-1, 0, 1],[-2, 0, 2],[-1, 0, 1]]) / (8 * cellsize)
    ky = np.array([[-1, -2, -1],[ 0,  0,  0],[ 1,  2,  1]]) / (8 * cellsize)

    # Petwitt風の二次微分カーネル
    kxx = np.array([[1, -2, 1],[1, -2, 1],[1, -2, 1]]) / (3* (cellsize ** 2))
    kyy = np.array([[1, 1, 1],[-2, -2, -2],[1, 1, 1]]) / (3 * (cellsize ** 2))
    kxy = np.array([[1, 0, -1],[0, 0, 0],[-1, 0, 1]]) / (4 * (cellsize ** 2))
    
    p, q, r, t, s = (np.sum(win * kernel, axis=(2, 3)) for kernel in [kx, ky, kxx, kyy, kxy])

    return p, q, r, t, s, cellsize, compute_mask, extent

def compute_profile_curvature(raster_path, pc_path, nodata_value):
      
    p, q, r, t, s, cellsize, mask, extent= compute_pqrst(raster_path, nodata_value)

    denom = ((p ** 2) + (q ** 2)) * ((1 + (p ** 2) + (q ** 2)) ** 1.5)
    denom = np.where(denom != 0, denom, 1)  # 0除算防止

    k = -((p ** 2 * r) + (2 * s * p * q) + (q ** 2 * t) ) / denom

    profile_curvature = np.where(mask, k, -9999)
    save_asc(profile_curvature, extent, cellsize, pc_path)

def compute_max_min_curvature(raster_path, maxc_path, minc_path, nodata_value):
    
    p, q, r, t, s, cellsize, mask, extent= compute_pqrst(raster_path, nodata_value)
    
    denom= (1 + (p ** 2) + (q ** 2))
    denom = np.where(denom != 0, denom, 1)  # 0除算防止
    mean = -(((1 + (q ** 2)) * r) - (2 * p * q * s) + ((1 + (p ** 2)) * t))/(2 * (denom ** 1.5))
    gau = (r * t - (s ** 2))/(denom ** 2)
    kmax = mean + (((mean ** 2) - gau)) ** 0.5
    kmin = mean - (((mean ** 2) - gau)) ** 0.5
    max_curvature = np.where(mask, kmax, -9999)
    min_curvature = np.where(mask, kmin, -9999)
    save_asc(max_curvature, extent, cellsize, maxc_path)
    save_asc(min_curvature, extent, cellsize, minc_path)
    
def compute_relative_elevation(raster_path, kernel_range, re_path, nodata_value):
    """
    セルサイズと実距離範囲（例：30m）に基づく相対標高差の計算
    """
    extent, _, _, cellsize, data = ras_info(raster_path)
    _, compute_mask = mask_com(data, nodata_value)
    
    pad_width = int(round(kernel_range / cellsize / 2))
    padded = np.pad(data, pad_width=pad_width, mode='edge')
    
    window_size = pad_width * 2 + 1
    
    win = sliding_window_view(padded, (window_size, window_size))

    # 最大 - 最小（NODATAは無視）
    win_nan = np.where(win == nodata_value, np.nan, win)
    center = win[:, :, pad_width, pad_width]
    min_val = np.nanmin(win_nan, axis=(2, 3))
    
    relative_elevation = np.where(compute_mask, center - min_val, nodata_value)
    save_asc(relative_elevation, extent, cellsize, re_path)
    
def compute_principal_curvature_orientation(raster_path, k1_path, k2_path, theta1_path, theta2_path, nodata_value):
    p, q, r, t, s, cellsize, mask, extent = compute_pqrst(raster_path, nodata_value)
    
    # 各成分の共通分母
    E = 1 + (p**2) + (q**2)
    E_3_2 = E ** 1.5

    # Shape operator 成分（要素単位）
    a = -(r * (1 + q**2) - 2 * p * q * s - t * p**2) / E_3_2
    b = -(s * (1 + q**2 - p**2) + p * q * (r - t)) / E_3_2
    d = -(t * (1 + p**2) - 2 * p * q * s - r * q**2) / E_3_2

    # 対称行列の構成
    P = np.empty((*a.shape, 2, 2))
    P[..., 0, 0] = a
    P[..., 0, 1] = b
    P[..., 1, 0] = b
    P[..., 1, 1] = d
    # 固有値・固有ベクトルを計算
    eigvals, eigvecs = np.linalg.eig(P)
    eigvals = eigvals.astype(np.float64)
    eigvecs = eigvecs.astype(np.float64)
    
    # 固有値のソート（大きい方を最大曲率に）
    idx = np.argsort(abs(eigvals), axis=-1)[..., ::-1]  # 降順

    k1 = np.take_along_axis(eigvals, idx[..., 0:1], axis=-1)[..., 0]

    k2 = np.take_along_axis(eigvals, idx[..., 1:2], axis=-1)[..., 0]

    # eigvecs: (..., 2, 2), idx: (..., 2)
    cond = idx[..., 0] == 0  # (...,) shape のブール配列

    v1 = np.where(cond[..., np.newaxis], eigvecs[..., 0, :], eigvecs[..., 1, :])  # (..., 2)
    v2 = np.where(cond[..., np.newaxis], eigvecs[..., 1, :], eigvecs[..., 0, :])  # (..., 2)

    theta1 = (90-np.degrees(np.arctan2(v1[..., 1], v1[..., 0]))) % 180

    theta2 = (90-np.degrees(np.arctan2(v2[..., 1], v2[..., 0]))) % 180

    # マスク適用
    k1 = np.where(mask, k1, nodata_value)
    k2 = np.where(mask, k2, nodata_value)
    theta1 = np.where(mask, theta1, nodata_value)
    theta2 = np.where(mask, theta2, nodata_value)

    # 保存
    save_asc(k1, extent, cellsize, k1_path)
    save_asc(k2, extent, cellsize, k2_path)
    save_asc(theta1, extent, cellsize, theta1_path)
    save_asc(theta2, extent, cellsize, theta2_path)

In [17]:
nodata_value = -9999
kernel_range = 30
base = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル"
for river in ["安部川","鬼怒川","大井川"]:
    for tile_id in tqdm(["01", "02", "03", "04"], total = 4):
        asc_path = fr"{base}\{river}\05m\elevation\{river}05m_{tile_id}.asc"
        gaussian_path = fr"{base}\{river}\05m\gaussian\{river}05m_{tile_id}_gau.asc"
        slope_path = fr"{base}\{river}\05m\slope\{river}05m_{tile_id}_slope_gau.asc"
        aspect_path = fr"{base}\{river}\05m\aspect\{river}05m_{tile_id}_aspect_gau.asc"
        aspect_diff_path = fr"{base}\{river}\05m\aspect\dif\{river}05m_{tile_id}_aspect_diff_gau.asc"
        pc_path = fr"{base}\{river}\05m\profilecurv\{river}05m_{tile_id}_profilecurv.asc"
        maxc_path = fr"{base}\{river}\05m\maxcurv\{river}05m_{tile_id}_maxcurv_gau.asc"
        minc_path = fr"{base}\{river}\05m\mincurv\{river}05m_{tile_id}_mincurv_gau.asc"
        re_path = fr"{base}\{river}\05m\relativeele\{river}05m_{tile_id}_relative_ele.asc"
        k1_path = fr"{base}\{river}\05m\curv\max\{river}05m_{tile_id}_curv_max.asc"
        k2_path = fr"{base}\{river}\05m\curv\min\{river}05m_{tile_id}_curv_min.asc"
        theta1_path = fr"{base}\{river}\05m\curv\max_theta\{river}05m_{tile_id}_curv_max_theta.asc"
        theta2_path = fr"{base}\{river}\05m\curv\min_theta\{river}05m_{tile_id}_curv_min_theta.asc"
        gaussian_smoothing(asc_path, gaussian_path, nodata_value)
        #compute_max_min_curvature(gaussian_path, maxc_path,minc_path, nodata_value)
        #compute_profile_curvature(asc_path, pc_path, nodata_value)
        #compute_principal_curvature_orientation(asc_path, k1_path, k2_path, theta1_path, theta2_path, nodata_value)
        compute_slope_aspect(gaussian_path, slope_path, aspect_path, aspect_diff_path, nodata_value)
        #compute_relative_elevation(asc_path, kernel_range, re_path, nodata_value)

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [02:22<00:00, 35.59s/it]


In [38]:
compute_slope_aspect(asc_path, slope_path, aspect_path, aspect_diff_path, nodata_value)
compute_profile_curvature(asc_path, pc_path, nodata_value)
compute_max_min_curvature(asc_path, maxc_path,minc_path nodata_value)
compute_relative_elevation(asc_path, kernel_range, re_path, nodata_value)
compute_principal_curvature_orientation(asc_path, k1_path, k2_path, theta1_path, theta2_path, nodata_value)

C:\Users\pc\AppData\Local\Temp\ipykernel_860\647707753.py:75: RuntimeWarning: All-NaN slice encountered
  min_diff = np.nanmin(diff, axis=0)
C:\Users\pc\AppData\Local\Temp\ipykernel_860\647707753.py:151: RuntimeWarning: All-NaN slice encountered
  max_val = np.nanmax(win_nan, axis=(2, 3))
C:\Users\pc\AppData\Local\Temp\ipykernel_860\647707753.py:152: RuntimeWarning: All-NaN slice encountered
  min_val = np.nanmin(win_nan, axis=(2, 3))


# Threshold

In [3]:
def ras_info(path):
    src = gdal.Open(path, gdal.GA_ReadOnly)
    left, xleng, rotx, top, roty, yleng = src.GetGeoTransform()
    
    xsize, ysize, cellsize = src.RasterXSize, src.RasterYSize, xleng
    
    botm = top + ysize * yleng
    right = left + xsize * xleng
    extent = [left, botm, right, top]
    
    data = src.GetRasterBand(1).ReadAsArray()
    return extent, xsize, ysize, cellsize, data

def save_asc(raster, extent, resolution, asc_path):

    rows, cols = raster.shape
    x_min, y_min = extent[0], extent[1]
    header = f"NCOLS {cols}\nNROWS {rows}\nXLLCORNER {x_min}\nYLLCORNER {y_min}\n"
    header += f"CELLSIZE {resolution}\nNODATA_VALUE 0\n"

    with open(asc_path, "w") as f:
        f.write(header)
        np.savetxt(f, raster, delimiter=" ")
def mask_com(data, nodata_value):
    
    data = np.pad(data, pad_width=1, mode='edge')
    windows = sliding_window_view(data, (3, 3))

    compute_mask = np.all(windows != nodata_value, axis=(2, 3))  
    return windows, compute_mask



In [ ]:
def compute_mean(raster_path, mean_path, nodata_value):
    extent, _, _, cellsize, data = ras_info(raster_path)
    mask = data<25
    data = np.where(mask, 0, data)
    win, compute_mask = mask_com(data, nodata_value)
    
    # Petwitt風の微分カーネル（一次微分）
    kernel = np.array([[1, 1, 1],[1, 1, 1],[1, 1, 1]]) / 9

    win, compute_mask = mask_com(data, nodata_value)
    mean = np.sum(win * kernel, axis=(2, 3))
    mean = np.where(mean<25, 0, mean)
    save_asc(mean, extent, cellsize, mean_path)

In [36]:
levee_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\levee\1\鬼怒川05m_03_levee1.asc"
elevation = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\asc\鬼怒川05m_03.asc"
levee_ele = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\levee\鬼怒川05m_03_test3.asc"
extent, _, _, cellsize, levee = ras_info(levee_path)
_, _, _, _, ele = ras_info(elevation)

tenba = np.where(levee != 0, ele, -9999)
#save_asc(tenba, extent, cellsize, levee_ele)

In [19]:
def save_asc(raster, extent, resolution, asc_path):

    rows, cols = raster.shape
    x_min, y_min = extent[0], extent[1]
    header = f"NCOLS {cols}\nNROWS {rows}\nXLLCORNER {x_min}\nYLLCORNER {y_min}\n"
    header += f"CELLSIZE {resolution}\nNODATA_VALUE 0\n"

    with open(asc_path, "w") as f:
        f.write(header)
        np.savetxt(f, raster, delimiter=" ")
nodata_value = -9999
kernel_range = 30
base = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル"
for river in ["大井川","鬼怒川", "安部川"]:
    for tile_id in ["01", "02", "03", "04"]:
        asc_path = fr"{base}\{river}\05m\elevation\{river}05m_{tile_id}.asc"
        slope_path = fr"{base}\{river}\05m\slope\{river}05m_{tile_id}_slope_gau.asc"
        aspect_path = fr"{base}\{river}\05m\aspect\{river}05m_{tile_id}_aspect.asc"
        aspect_diff_path = fr"{base}\{river}\05m\aspect\dif\{river}05m_{tile_id}_aspect_diff_gau.asc"
        pc_path = fr"{base}\{river}\05m\profilecurv\{river}05m_{tile_id}_profilecurv.asc"
        maxc_path = fr"{base}\{river}\05m\maxcurv\{river}05m_{tile_id}_maxcurv_gau.asc"
        minc_path = fr"{base}\{river}\05m\mincurv\{river}05m_{tile_id}_mincurv.asc"
        re_path = fr"{base}\{river}\05m\relativeele\{river}05m_{tile_id}_relative_ele.asc"
        k1_path = fr"{base}\{river}\05m\curv\max\{river}05m_{tile_id}_curv_max.asc"
        k2_path = fr"{base}\{river}\05m\curv\min\{river}05m_{tile_id}_curv_min.asc"
        theta1_path = fr"{base}\{river}\05m\curv\max_theta\{river}05m_{tile_id}_curv_max_theta.asc"
        theta2_path = fr"{base}\{river}\05m\curv\min_theta\{river}05m_{tile_id}_curv_min_theta.asc"
        levee_path = fr"{base}\{river}\05m\levee\{river}05m_{tile_id}_levee_gau.asc"
        extent, _, _, cellsize, aspect = ras_info(aspect_diff_path)
        _, _, _, _, maxcurv = ras_info(maxc_path)
        _, _, _, _, relative_ele = ras_info(re_path)
        _, _, _, _, slope = ras_info(slope_path)
        re_list = range(1, 8)
        sl_list = range(5, 19, 2)
        ap_list = range(10, 50, 5)
        pc_list = np.arange(1, 9, 1)* 0.02

        leveeas = 0
        for re in tqdm(re_list, total = len(re_list)):
            for sl in sl_list:
                for ap in ap_list:
                    for pc in pc_list:
                        mask_re = (relative_ele!= -9999)&(relative_ele<8)&(relative_ele>re)
                        levee = np.where(mask_re, 1, 0)
                        mask_sl = (levee!= 0)&(slope<sl)
                        levee1 = np.where(mask_sl, 1, 0)
                        mask_as = (levee1!= 0)&(aspect<ap)
                        levee2 = np.where(mask_as, 1, 0)
                        mask_pc = (levee2!= 0)&(maxcurv>pc)
                        levee3 = np.where(mask_pc, 1, 0)
                        leveeas +=levee3

        save_asc(leveeas, extent, cellsize, levee_path)

100%|███████████████████████████████████████████████████████████████████████████████████| 7/7 [16:50<00:00, 144.29s/it]


In [13]:
nodata_value = -9999
kernel_range = 30
base = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル"
for river in ["安部川"]:
    for tile_id in ["01", "02", "03", "04"]:
        asc_path = fr"{base}\{river}\05m\elevation\{river}05m_{tile_id}.asc"
        gaussian_path = fr"{base}\{river}\05m\gaussian\{river}05m_{tile_id}_gau.asc"
        slope_path = fr"{base}\{river}\05m\slope\{river}05m_{tile_id}_slope_gau.asc"
        aspect_diff_path = fr"{base}\{river}\05m\aspect\dif\{river}05m_{tile_id}_aspect_diff_gau.asc"
        pc_path = fr"{base}\{river}\05m\profilecurv\{river}05m_{tile_id}_profilecurv_gau.asc"
        maxc_path = fr"{base}\{river}\05m\maxcurv\{river}05m_{tile_id}_maxcurv_gau.asc"
        minc_path = fr"{base}\{river}\05m\mincurv\{river}05m_{tile_id}_mincurv_gau.asc"
        re_path = fr"{base}\{river}\05m\relativeele\{river}05m_{tile_id}_relative_ele.asc"
        k1_path = fr"{base}\{river}\05m\curv\max\{river}05m_{tile_id}_curv_max_gau.asc"
        k2_path = fr"{base}\{river}\05m\curv\min\{river}05m_{tile_id}_curv_min_gau.asc"
        theta1_path = fr"{base}\{river}\05m\curv\max_theta\{river}05m_{tile_id}_curv_max_theta_gau.asc"
        theta2_path = fr"{base}\{river}\05m\curv\min_theta\{river}05m_{tile_id}_curv_min_theta_gau.asc"
        levee_path = fr"{base}\{river}\05m\levee\{river}05m_{tile_id}_levee_gau.asc"
        extent, _, _, cellsize, aspect = ras_info(aspect_diff_path)
        _, _, _, _, maxcurv = ras_info(k1_path)
        _, _, _, _, relative_ele = ras_info(re_path)
        _, _, _, _, slope = ras_info(slope_path)
        re_list = range(1, 8)
        sl_list = range(5, 19, 2)
        ap_list = range(10, 50, 5)
        pc_list = np.arange(1, 9, 1)* 0.02

        leveeas = 0
        for re in tqdm(re_list, total = len(re_list)):
            for sl in sl_list:
                for ap in ap_list:
                    for pc in pc_list:
                        mask_re = (relative_ele!= -9999)&(relative_ele<8)&(relative_ele>re)
                        levee = np.where(mask_re, 1, 0)
                        mask_sl = (levee!= 0)&(slope<sl)
                        levee1 = np.where(mask_sl, 1, 0)
                        mask_as = (levee1!= 0)&(aspect<ap)
                        levee2 = np.where(mask_as, 1, 0)
                        mask_pc = (levee2!= 0)&(maxcurv>pc)
                        levee3 = np.where(mask_pc, 1, 0)
                        leveeas +=levee3

        save_asc(leveeas, extent, cellsize, levee_path)

100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [11:35<00:00, 99.42s/it]


### 本家

In [42]:
re_list = range(1, 8)
sl_list = range(5, 19, 2)
ap_list = range(10, 50, 5)
pc_list = np.arange(1, 9, 1)* 0.02

leveeas = 0
for re in tqdm(re_list, total = len(re_list)):
    for sl in sl_list:
        for ap in ap_list:
            for pc in pc_list:
                mask_re = (relative_ele!= -9999)&(relative_ele<8)&(relative_ele>re)
                levee = np.where(mask_re, 1, 0)
                mask_sl = (levee!= 0)&(slope<sl)
                levee1 = np.where(mask_sl, 1, 0)
                mask_as = (levee1!= 0)&(aspect<ap)
                levee2 = np.where(mask_as, 1, 0)
                mask_pc = (levee2!= 0)&(maxcurv>pc)
                levee3 = np.where(mask_pc, 1, 0)
                leveeas +=levee3
                
save_asc(leveeas, extent, cellsize, levee_path)

100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [10:00<00:00, 85.84s/it]


### 

In [4]:
slope_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\slope\鬼怒川05m_03_slope.asc"
aspect_diff_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\aspect\dif\鬼怒川05m_03_aspect_diff.asc"
maxc_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\maxcurv\鬼怒川05m_03_maxcurv.asc"
re_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\relativeele\鬼怒川05m_03_relative_ele.asc"
levee_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\levee\1\鬼怒川05m_03_levee1_test.asc"
extent, _, _, cellsize, aspect = ras_info(aspect_diff_path)
_, _, _, _, maxcurv = ras_info(maxc_path)
_, _, _, _, relative_ele = ras_info(re_path)
_, _, _, _, slope = ras_info(slope_path)

C:\Users\pc\miniconda3\envs\pynote\Lib\site-packages\osgeo\gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [9]:
levee_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\levee\1\鬼怒川05m_03_levee1_test.asc"

In [10]:
mask_as1 = (aspect!= -9999)&(aspect<22.5)
kernel = np.array([[1, 1, 1],[1, 1, 1],[1, 1, 1]]) / 9
win, compute_mask = mask_com(mask_as1, -9999)
mask_as2 = np.sum(win * kernel, axis=(2, 3))
mask_as = np.where((mask_as2!=0)&(aspect<45), 1, 0)

levee = np.where(mask_as, 1, 0)

mask_sl1 = (levee!= 0)&(slope < 10)
#win, compute_mask = mask_com(mask_sl1, -9999)
#mask_sl2 = np.sum(win * kernel, axis=(2, 3))
mask_sl = np.where((mask_sl1!=0)&(slope<17), 1, 0)
levee1 = np.where(mask_sl, 1, 0)

mask_pc1 = (levee1!= 0)&(maxcurv>0.02)
win, compute_mask = mask_com(mask_pc1, -9999)
mask_pc2 = np.sum(win * kernel, axis=(2, 3))
mask_pc = np.where((mask_pc2!=0)&(maxcurv>0.01), 1, 0)
levee2 = np.where(mask_pc, 1, 0)

mask_re = (levee2!= 0)&(relative_ele<7.5)&(relative_ele>1.0)
levee3 = np.where(mask_re, 1, 0)

save_asc(levee3, extent, cellsize, levee_path)

In [101]:
k = 0.02
R = 1/k
h = (R) - np.sqrt(R**2 - 32.5**2)
h

np.float64(12.003289616073339)

In [ ]:
def ras_info(path):
    src = gdal.Open(path, gdal.GA_ReadOnly)
    left, xleng, rotx, top, roty, yleng = src.GetGeoTransform()
    
    xsize, ysize, cellsize = src.RasterXSize, src.RasterYSize, xleng
    
    botm = top + ysize * yleng
    right = left + xsize * xleng
    extent = [left, botm, right, top]
    
    data = src.GetRasterBand(1).ReadAsArray()
    return extent, xsize, ysize, cellsize, data

def save_asc(raster, extent, resolution, asc_path):
    rows, cols = raster.shape
    
    x_min, y_min = extent[0], extent[1]
    header = f"NCOLS {cols}\nNROWS {rows}\nXLLCORNER {x_min}\nYLLCORNER {y_min}\n"
    header += f"CELLSIZE {resolution}\nNODATA_VALUE 0\n"

    with open(asc_path, "w") as f:
        f.write(header)
        np.savetxt(f, raster, delimiter=" ", fmt="%.4f")

def mask_com(data, nodata_value):
    
    data = np.pad(data, pad_width=1, mode='edge')
    windows = sliding_window_view(data, (3, 3))

    compute_mask = np.all(windows != nodata_value, axis=(2, 3))  
    return windows, compute_mask

def gaussian_kernel_3x3(sigma=0.8493):

    ax = np.arange(-1, 2)
    xx, yy = np.meshgrid(ax, ax)
    kernel = 1/(2*np.pi**2)*np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel /= np.sum(kernel)
    return np.round(kernel, 4)

def gaussian_smoothing(data, nodata_value, sigma=0.8493):

    kernel = gaussian_kernel_3x3(sigma)

    win, compute_mask = mask_com(data, nodata_value)
    gaussian = np.sum(win * kernel, axis=(2, 3))
    gaussian = np.where(compute_mask, gaussian, -9999)
    return gaussian
    
def compute_slope_aspect(data, cellsize, nodata_value):

    win, compute_mask = mask_com(data, nodata_value)
    
    kx = np.array([[-1, 0, 1],[-2, 0, 2],[-1, 0, 1]]) / (8 * cellsize)
    ky = np.array([[-1, -2, -1],[ 0, 0, 0],[ 1, 2, 1]]) / (8 * cellsize)

    dz_dx, dz_dy = (np.sum(win * kernel, axis=(2, 3)) for kernel in [kx, ky])

    slope_deg, aspect_deg = (np.full_like(dz_dx, nodata_value, dtype=float) for _ in range(2))
    
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg[compute_mask] = np.degrees(slope_rad[compute_mask])
    
    aspect_rad = np.atan2(dz_dy, -dz_dx)
    aspect_deg[compute_mask] = (90 - np.degrees(aspect_rad[compute_mask])) % 360
    
    flat_mask = (dz_dx == 0) & (dz_dy == 0) & compute_mask
    aspect_deg[flat_mask] = -1
    
    winas, _ = mask_com(aspect_deg, nodata_value)
    dir_pairs = [((1, 2), (1, 0)),  # E-W
                 ((2, 1), (0, 1)),  # S-N
                 ((0, 2), (2, 0)),  # NE-SW
                 ((2, 2), (0, 0))]  # SE-NW

    pair1, pair2 = zip(*[
        (winas[:, :, y1, x1], winas[:, :, y2, x2])
        for (y1, x1), (y2, x2) in dir_pairs
    ])
    pair1, pair2 = (np.stack(pair, axis=0) for pair in [pair1, pair2])

    pair1_opposite = (pair1 + 180) % 360
    diff = np.abs(pair1_opposite - pair2)% 360
    diff = np.minimum(diff, 360 - diff)
    invalid_mask = (pair1 == nodata_value) | (pair2 == nodata_value)
    diff = np.where(invalid_mask, np.nan, diff)
    min_diff = np.nanmin(diff, axis=0)
    mask = (compute_mask) | ((dz_dx != 0) | (dz_dy != 0))
    min_diff = np.where(mask, min_diff, nodata_value)
    return slope_deg, min_diff

def compute_pqrst(data, cellsize, nodata_value):
    
    win, compute_mask = mask_com(data, nodata_value)
    
    kx = np.array([[-1, 0, 1],[-2, 0, 2],[-1, 0, 1]]) / (8 * cellsize)
    ky = np.array([[-1, -2, -1],[ 0,  0,  0],[ 1,  2,  1]]) / (8 * cellsize)

    # Petwitt風の二次微分カーネル
    kxx = np.array([[1, -2, 1],[1, -2, 1],[1, -2, 1]]) / (3* (cellsize ** 2))
    kyy = np.array([[1, 1, 1],[-2, -2, -2],[1, 1, 1]]) / (3 * (cellsize ** 2))
    kxy = np.array([[1, 0, -1],[0, 0, 0],[-1, 0, 1]]) / (4 * (cellsize ** 2))
    
    p, q, r, t, s = (np.sum(win * kernel, axis=(2, 3)) for kernel in [kx, ky, kxx, kyy, kxy])

    return p, q, r, t, s, cellsize, compute_mask, extent
    
def compute_principal_curvature_orientation(data, cellsize, nodata_value):
    
    p, q, r, t, s, cellsize, mask, extent = compute_pqrst(data, cellsize, nodata_value)
    
    E = 1 + (p**2) + (q**2)
    E_3_2 = E ** 1.5

    # Shape operator 成分（要素単位）
    a = -(r * (1 + q**2) - 2 * p * q * s - t * p**2) / E_3_2
    b = -(s * (1 + q**2 - p**2) + p * q * (r - t)) / E_3_2
    d = -(t * (1 + p**2) - 2 * p * q * s - r * q**2) / E_3_2


    P = np.empty((*a.shape, 2, 2))
    P[..., 0, 0] = a
    P[..., 0, 1] = b
    P[..., 1, 0] = b
    P[..., 1, 1] = d

    eigvals, eigvecs = np.linalg.eig(P)
    eigvals = eigvals.astype(np.float64)
    eigvecs = eigvecs.astype(np.float64)
    
    idx = np.argsort(abs(eigvals), axis=-1)[..., ::-1]  # 降順

    k1 = np.take_along_axis(eigvals, idx[..., 0:1], axis=-1)[..., 0]

    k2 = np.take_along_axis(eigvals, idx[..., 1:2], axis=-1)[..., 0]

    cond = idx[..., 0] == 0  # (...,) shape のブール配列

    v1 = np.where(cond[..., np.newaxis], eigvecs[..., 0, :], eigvecs[..., 1, :])  # (..., 2)
    v2 = np.where(cond[..., np.newaxis], eigvecs[..., 1, :], eigvecs[..., 0, :])  # (..., 2)

    theta1 = (90-np.degrees(np.arctan2(v1[..., 1], v1[..., 0]))) % 180

    theta2 = (90-np.degrees(np.arctan2(v2[..., 1], v2[..., 0]))) % 180

    # マスク適用
    k1 = np.where(mask, k1, nodata_value)
    k2 = np.where(mask, k2, nodata_value)
    theta1 = np.where(mask, theta1, nodata_value)
    theta2 = np.where(mask, theta2, nodata_value)

    return k1

In [ ]:
nodata_value = -9999
sigma_list = np.arange(10, 21, 1)* 0.05
base = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル"
for river in ["鬼怒川", "安部川"]:
    for tile_id in ["01", "02", "03", "04"]:
        asc_path = fr"{base}\{river}\05m\elevation\{river}05m_{tile_id}.asc"
        re_path = fr"{base}\{river}\05m\relativeele\{river}05m_{tile_id}_relative_ele.asc"

        levee_path = fr"{base}\{river}\05m\levee\1\{river}05m_{tile_id}_levee1.asc"

        extent, _, _, cellsize, data = ras_info(asc_path)
        _, _, _, _, relative_ele = ras_info(re_path)
        for sigma in sigma_list:
            gaussian = gaussian_smoothing(data, nodata_value, sigma)    
            slope, aspect = compute_slope_aspect(gaussian, cellsize, nodata_value)
            maxcurv = compute_principal_curvature_orientation(gaussian, cellsize, nodata_value)
            mask_re = (relative_ele!= -9999)&(relative_ele<8)&(relative_ele>1)
            levee = np.where(mask_re, 1, 0)
            mask_sl = (levee!= 0)&(slope<17)
            levee1 = np.where(mask_sl, 1, 0)
            mask_as = (levee1!= 0)&(aspect<45)
            levee2 = np.where(mask_as, 1, 0)
            mask_pc = (levee2!= 0)&(maxcurv>0.02)
            levee3 = np.where(mask_pc, 1, 0)
            leveeas +=levee3
save_asc(leveeas, extent, cellsize, levee_path)

# mask

In [39]:
levee_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\鬼怒川\05m\levee\鬼怒川05m_03_test4.asc"

mask_re = (relative_ele!= -9999)&(relative_ele<15)&(relative_ele>1)
levee = np.where(mask_re, 1, 0)

mask_sl = (levee!= 0)&(slope<17)
levee1 = np.where(mask_sl, 1, 0)

mask_as1 = (levee1!= 0)&(aspect<45)
kernel = np.array([[1, 1, 1],[1, 1, 1],[1, 1, 1]]) / 9
win, compute_mask = mask_com(mask_as1, -9999)
mask_as2 = np.sum(win * kernel, axis=(2, 3))
mask_as = np.where(mask_as2!=0, 1, 0)
levee2 = np.where(mask_as, 1, 0)
mask_pc = (levee2!= 0)&(maxcurv>0.02)
levee3 = np.where(mask_pc, 1, 0)

save_asc(levee2, extent, cellsize, levee_path)

In [23]:
slope_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\安部川水系\05m\slope\安部川05m_01_slope.asc"
aspect_diff_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\安部川水系\05m\aspect\dif\安部川05m_01_aspect_diff.asc"
maxc_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\安部川水系\05m\maxcurv\安部川05m_01_maxcurv.asc"
re_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\安部川水系\05m\relativeele\安部川05m_01_relative_ele.asc"

extent, _, _, cellsize, aspect = ras_info(aspect_diff_path)
_, _, _, _, maxcurv = ras_info(maxc_path)
_, _, _, _, relative_ele = ras_info(re_path)
_, _, _, _, slope = ras_info(slope_path)

re_list = range(1, 8)
sl_list = range(5, 19, 2)
ap_list = range(1, 25, 3)
pc_list = np.arange(1, 9, 1)* 0.02
asc_base = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\安部川水系\05m\test\\"
for re in re_list:
    rasc_path = asc_base + f"relative_ele_{re:02d}.asc"
    mask_re = (relative_ele!= -9999)&(relative_ele<15)&(relative_ele>re)
    levee = np.where(mask_re, 1, 0)
    save_asc(levee, extent, cellsize, rasc_path)
for sl in sl_list:
    rasc_path = asc_base + f"slope_{sl:02d}.asc"
    mask_sl = (slope!= -9999)&(slope<sl)
    levee = np.where(mask_sl, 1, 0)
    save_asc(levee, extent, cellsize, rasc_path)
for ap in ap_list:
    rasc_path = asc_base + f"aspect_diff_{ap:02d}.asc"
    mask_as = (aspect!= -9999)&(aspect<ap)
    levee = np.where(mask_as, 1, 0)
    save_asc(levee, extent, cellsize, rasc_path)
for pc in pc_list:
    pc_val = int(pc * 100)  # 例: 0.02 → 2
    rasc_path = asc_base + f"maxcurv_{pc_val:02d}.asc"
    mask_pc = (maxcurv!= -9999)&(maxcurv<pc)
    levee = np.where(mask_pc, 1, 0)
    save_asc(levee, extent, cellsize, rasc_path)

ValueError: operands could not be broadcast together with shapes (1851,2282) (1853,2287) 

In [8]:
levee = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\安部川水系\05m\levee\安部川05m_02_levee.asc"

extent, xsize, ysize, cellsize, data = ras_info(levee)

In [9]:
test = np.where(data!= 0, 1, 0)

In [10]:
np.sum(test)

np.int64(20559)

## Noise Remove

In [24]:
def ras_info(path):
    src = gdal.Open(path, gdal.GA_ReadOnly)
    left, xleng, rotx, top, roty, yleng = src.GetGeoTransform()
    
    xsize, ysize, cellsize = src.RasterXSize, src.RasterYSize, xleng
    
    botm = top + ysize * yleng
    right = left + xsize * xleng
    extent = [left, botm, right, top]
    
    data = src.GetRasterBand(1).ReadAsArray()
    return extent, xsize, ysize, cellsize, data

def save_asc(raster, extent, resolution, asc_path):

    rows, cols = raster.shape
    x_min, y_min = extent[0], extent[1]
    header = f"NCOLS {cols}\nNROWS {rows}\nXLLCORNER {x_min}\nYLLCORNER {y_min}\n"
    header += f"CELLSIZE {resolution}\nNODATA_VALUE 0\n"

    with open(asc_path, "w") as f:
        f.write(header)
        np.savetxt(f, raster, delimiter=" ", fmt="%.4f")

def mask_com(data, nodata_value):
    
    data = np.pad(data, pad_width=1, mode='edge')
    windows = sliding_window_view(data, (3, 3))

    compute_mask = np.all(windows != nodata_value, axis=(2, 3))  
    return windows, compute_mask


def compute_mean(raster_path, mean_path, nodata_value):
    extent, _, _, cellsize, data = ras_info(raster_path)
    mask = data<25
    data = np.where(mask, 0, data)
    win, compute_mask = mask_com(data, nodata_value)
    
    # Petwitt風の微分カーネル（一次微分）
    kernel = np.array([[1, 1, 1],[1, 1, 1],[1, 1, 1]]) / 9

    win, compute_mask = mask_com(data, nodata_value)
    mean = np.sum(win * kernel, axis=(2, 3))
    mean = np.where(mean<25, 0, mean)

    win, compute_mask = mask_com(mean, -9999)
    win_nan = np.where(win == nodata_value, np.nan, win)
    med_val = np.nanmedian(win_nan, axis=(2, 3))

    mask_med = med_val == 0
    output = np.where(mask_med, 0, data)
    save_asc(output, extent, cellsize, mean_path)

In [28]:
levee = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\大井川\05m\levee\大井川05m_04_levee.asc"
mean_path = r"C:\Users\pc\note\河川堤防自動抽出\数値標高モデル\大井川\05m\levee\大井川05m_04_levee_c.asc"
compute_mean(levee, mean_path, -9999)

## other

In [22]:
def ras_info(path):
    src = gdal.Open(path, gdal.GA_ReadOnly)
    left, xleng, rotx, top, roty, yleng = src.GetGeoTransform()
    
    xsize, ysize, cellsize = src.RasterXSize, src.RasterYSize, xleng
    
    botm = top + ysize * yleng
    right = left + xsize * xleng
    extent = [left, botm, right, top]
    
    data = src.GetRasterBand(1).ReadAsArray()
    return extent, xsize, ysize, cellsize, data

def save_asc(raster, extent, resolution, asc_path):

    rows, cols = raster.shape
    x_min, y_min = extent[0], extent[1]
    header = f"NCOLS {cols}\nNROWS {rows}\nXLLCORNER {x_min}\nYLLCORNER {y_min}\n"
    header += f"CELLSIZE {resolution}\nNODATA_VALUE 0\n"

    with open(asc_path, "w") as f:
        f.write(header)
        np.savetxt(f, raster, delimiter=" ", fmt="%.4f")

def mask_com(data, nodata_value):
    
    data = np.pad(data, pad_width=1, mode='edge')
    windows = sliding_window_view(data, (3, 3))

    compute_mask = np.all(windows != nodata_value, axis=(2, 3))  
    return windows, compute_mask


def compute_mean(raster_path, mean_path, nodata_value):
    extent, _, _, cellsize, data = ras_info(raster_path)
    mask = data<25
    data = np.where(mask, 0, data)
    win, compute_mask = mask_com(data, nodata_value)
    
    # Petwitt風の微分カーネル（一次微分）
    kernel = np.array([[1, 1, 1],[1, 1, 1],[1, 1, 1]]) / 9

    win, compute_mask = mask_com(data, nodata_value)
    mean = np.sum(win * kernel, axis=(2, 3))

    save_asc(mean, extent, cellsize, mean_path)

### simple feature

In [ ]:
angle_t = 5
distance_t = 10000
touei = 'PROJCS["WGS_1984_Web_Mercator_Auxiliary_Sphere",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Mercator_Auxiliary_Sphere"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",0.0],PARAMETER["Standard_Parallel_1",0.0],PARAMETER["Auxiliary_Sphere_Type",0.0],UNIT["Meter",1.0]]'
shp_path = r"C:\Users\pc\note\河川堤防自動抽出\平面直角座標\平面直角座標.shp"

def detect_encoding(dbf_path):
    with open(dbf_path, "rb") as f:
        raw_data = f.read(10000)  
        result = chardet.detect(raw_data)
    return result['encoding']

def distance_point_to_line(x0, y0, x1, y1, x2, y2):
    """
    点 (x0, y0) が直線 (x1, y1) - (x2, y2) からどれだけ離れているかを計算
    """
    if x1 == x2:  # 垂直な直線の場合
        return abs(x0 - x1)
    else:
        a = (y2 - y1) / (x2 - x1)
        b = y1 - a * x1
        return abs(a * x0 - y0 + b) / ((a**2 + 1) ** 0.5)

def calculate_angle(p1, p2, p3):
    """
    3点 (p1, p2, p3) を通る角度変化を計算
    p1 → p2 → p3 の角度差を [-180, 180] の範囲で返す
    """
    v1 = (p1[0] - p2[0], p1[1] - p2[1])  # ベクトル p2→p1
    v2 = (p3[0] - p2[0], p3[1] - p2[1])  # ベクトル p2→p3

    dot = v1[0] * v2[0] + v1[1] * v2[1]  # 内積
    det = v1[0] * v2[1] - v1[1] * v2[0]  # 外積

    angle = math.atan2(det, dot)  # 角度 (ラジアン)
    return math.degrees(angle)  # 角度 (度数法)

def simplify_polyline(points, angle_threshold, buffer_dist):
    """
    角度と距離の条件を両方満たす場合に点を保持するポリライン簡略化
    :param points: ポリラインの座標点 [(x1, y1), (x2, y2), ..., (xn, yn)]
    :param angle_threshold: 角度の閾値（この値以上の角度変化がある場合のみ保持）
    :param buffer_dist: バッファ距離（閾値、線分からの距離がこれ以上なら保持）
    :return: 簡略化されたポリラインの座標点リスト
    """
    if len(points) < 3:
        return points  # 3点未満ならそのまま

    simplified = [points[0]]  # 最初の点を追加
    p1 = points[0]  # 始点を固定
    i = 1  # p2 のインデックス

    while i < len(points) - 1:
        p2 = points[i]
        p3 = points[i + 1]

        angle = calculate_angle(p1, p2, p3)
        distance = distance_point_to_line(p2[0], p2[1], p1[0], p1[1], p3[0], p3[1])

        # 角度が閾値以上 OR 距離が閾値以上なら p2 を保持
        if abs(angle) - 180 < angle_threshold * (-1) or distance > buffer_dist:
            simplified.append(p2)  # 判定を満たした点を追加
            p1 = p2  # 次の始点を更新

        i += 1  # 次の判定へ進む

    simplified.append(points[-1])  # 最後の点を追加
    return simplified


dbf_path = shp_path.replace(".shp", ".dbf")
cpg_path = shp_path.replace(".shp", ".cpg")
prj_path = shp_path.replace(".shp", ".prj")
if os.path.exists(cpg_path):
    with open(cpg_path, "r") as cpg_file:
        encoding = cpg_file.read().strip()
    cpg_file.close()
else:
    encoding = detect_encoding(dbf_path)
    
if os.path.exists(prj_path):
    with open(prj_path, "r") as prj_file:
        prjj = prj_file.read().strip()
    prj_file.close()
src = shapefile.Reader(shp_path, encoding = encoding)  
sf = src.shapes()  
rf = src.records()  

fields = src.fields[1:] 
fn = [field[0] for field in fields]
fty = [field[1] for field in fields]
fsi = [field[2] for field in fields]
fde = [field[3] for field in fields]
new_row = []

for shp, rec in tqdm(zip(sf, rf), total=len(sf)):

    sub_geo = []
    geo = []

    parts = list(shp.parts) + [len(shp.points)]  # 終端インデックスを追加

    for i in range(len(parts) - 1):
        start_idx = parts[i]
        end_idx = parts[i + 1]

        sub_geo = []
        for j in range(start_idx, end_idx):
            if os.path.exists(prj_path):
                if prjj == touei:
                    x, y = shp.points[j]
                else:
                    crsp = CRS(prjj)
                    crst = CRS(touei)
                    epsg_codep = "epsg:" + crsp.to_authority(min_confidence=50)[1]
                    epsg_codet = "epsg:" + crst.to_authority(min_confidence=50)[1]
                    transformer = pyproj.Transformer.from_crs(epsg_codep, epsg_codet, always_xy=True)
                    x, y = transformer.transform(*shp.points[j])
            else:
                x, y = shp.points[j]

            sub_geo.append((x, y))

        geo.append(sub_geo)  # パートごとに追加

    # 属性と一緒に行データを格納
    row_data = {"geometry": geo}
    for field_name in fn:
        row_data[field_name] = rec[field_name]
    new_row.append(row_data)
nf = pd.DataFrame(new_row)
# DataFrame作成
nr = []
dd = 0

for j in tqdm(range(len(nf)), total=len(nf)):
    parts = nf["geometry"][j]  # 全パート（リストのリスト）

    simplified_parts = []
    dec = 0  # この行で削減された点数

    for part in parts:
        simplified = simplify_polyline(part, angle_t, distance_t)
        dec += len(part) - len(simplified)
        simplified_parts.append(simplified)

    dd += dec

    row_data = {"geometry": simplified_parts}
    for field_name in fn:
        row_data[field_name] = nf[field_name][j]
    nr.append(row_data)

nnf = pd.DataFrame(nr)
rpath = shp_path.replace(".shp", "_simple.shp")
with shapefile.Writer(rpath, shapeType=st) as w:
    
    for field_name, ty, si, de in zip(fn, fty, fsi, fde):
        w.field(field_name, ty, si, de)
         
    total = len(nnf)

    for i, row in tqdm(enumerate(nnf.itertuples(index=False), start=1), total = len(nnf)):
        if st == shapefile.POLYLINE:
            w.line(row.geometry)  # ラインデータの追加（リストで囲む必要あり）
        else:
            w.poly(row.geometry)
        w.record(*[getattr(row, field) for field in fn])  # 属性データを追加

w.close()

rprj = rpath.replace("shp", "prj")

with open(rprj,  'w') as prj:

    prj.write(touei)

prj.close()
    

print(f"{dd}個の座標が減りました.")

### gaussian

In [ ]:
import numpy as np

def gaussian_smoothing_nodata(data, nodata_value=-9999):
    """
    NoData対応 3x3 ガウシアンフィルター（NumPyのみ）

    Parameters:
        data (ndarray): 入力2D配列
        nodata_value (float): 無効値とみなす値

    Returns:
        smoothed (ndarray): 平滑化後の配列
    """
    # ガウシアンカーネル（σ≈1, 正規化済み）
    kernel = np.array([
        [1, 2, 1],
        [2, 4, 2],
        [1, 2, 1]
    ], dtype=float)
    kernel /= kernel.sum()

    # 元の関数構造を使う
    data = np.pad(data, pad_width=1, mode='edge')
    windows = np.lib.stride_tricks.sliding_window_view(data, (3, 3))
    valid_mask = windows != nodata_value
    center = windows[:, :, 1, 1]
    valid_count = np.sum(valid_mask, axis=(2, 3))

    compute_mask = (valid_count >= 7) & (center != nodata_value)
    valid_kernel = np.where(valid_mask, kernel, 0)
    weight = np.sum(valid_kernel, axis=(2, 3)).astype(float)
    weighted_sum = np.sum(windows * valid_kernel, axis=(2, 3))

    smoothed = np.full_like(weight, nodata_value, dtype=float)
    np.divide(weighted_sum, weight, where=compute_mask, out=smoothed)

    return smoothed

#data1 = gaussian_smoothing_nodata(data, nodata_value=-9999)

### after

In [ ]:
#aspect, slope
def compute_weighted_gradient(data, kernel, nodata_value=-9999):
    """
    加重カーネルを用いた勾配計算（NoDataおよび有効ピクセル数の条件付き）
    """
    windows, valid_mask, compute_mask = mask_com(data, nodata_value)
    valid_kernel = np.where(valid_mask, kernel, 0)
    weight = np.sum(valid_kernel, axis=(2, 3)).astype(float)
    weighted_sum = np.sum(windows * valid_kernel, axis=(2, 3)) * 4
    gradient = np.full_like(weight, nodata_value, dtype=float)
    np.divide(weighted_sum, weight, where=compute_mask, out=gradient)
    return gradient,compute_mask

def compute_slope(raster_path, cellsize=1.0, nodata_value=-9999):
    """
    重み付き勾配から傾斜角（°）を計算

    Parameters:
        data (ndarray): 入力 2D 配列（標高）
        cellsize (float): セル幅（m）
        nodata_value (float/int): 無効値とみなす値

    Returns:
        slope_deg (ndarray): 傾斜角（°）を格納した配列
    """
    # 勾配計算用カーネル
    extent, xsize, ysize, cellsize, data = ras_info(raster_path)
    kernel_dx_pos = np.array([[0, 0, 1], [0, 0, 2], [0, 0, 1]])
    kernel_dx_neg = np.array([[1, 0, 0], [2, 0, 0], [1, 0, 0]])
    kernel_dy_pos = np.array([[0, 0, 0], [0, 0, 0], [1, 2, 1]])
    kernel_dy_neg = np.array([[1, 2, 1], [0, 0, 0], [0, 0, 0]])

    dz_dx1, compute_mask = compute_weighted_gradient(data, kernel_dx_pos, nodata_value)
    dz_dx2, _= compute_weighted_gradient(data, kernel_dx_neg, nodata_value)
    dz_dy1, _ = compute_weighted_gradient(data, kernel_dy_pos, nodata_value)
    dz_dy2, _ = compute_weighted_gradient(data, kernel_dy_neg, nodata_value)

    # X, Y 勾配を合成
    dz_dx = np.where(compute_mask, (dz_dx1 - dz_dx2) / (8 * cellsize), nodata_value)
    dz_dy = np.where(compute_mask, (dz_dy1 - dz_dy2) / (8 * cellsize), nodata_value)
    # 傾斜角（ラジアン → 度）
    slope_deg = np.full_like(dz_dx, nodata_value, dtype=float)
    aspect_deg = np.full_like(dz_dx, nodata_value, dtype=float)
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg[compute_mask] = np.degrees(slope_rad[compute_mask])
    aspect_rad = np.atan2(dz_dy, -dz_dx)
    aspect_deg[compute_mask] = (90 - np.degrees(aspect_rad[compute_mask])) % 360
    flat_mask = (dz_dx == 0) & (dz_dy == 0) & compute_mask
    aspect_deg[flat_mask] = -1
    return slope_deg, aspect_deg, compute_mask

#使用例
#slope, aspect, mask= compute_slope(data, cellsize=cellsize, nodata_value=-9999)

#profile curvature

### gee

In [12]:
import sympy

sympy.init_printing()

sympy.var("d Z1 Z3 Z6 Z4 Z9 Z7")

p1 = d**2*d*d*(d+d)*(Z3-Z1)
p2 = d*(d**2*d**2+d**2*d**2)*(Z6-Z4)
p3 = d*d**2*d*(d+d)*(Z9-Z7)
p4 = p1+p2+p3

p5 = 2
p6 = d**2*d**2*(d+d)**2
p7 = d**2*(d**2*d**2+d**2*d**2)
p8 = (p6+p7)*p5

p = p4/p8
pk = sympy.simplify(p)
display(pk)

-Z₁ + Z₃ - Z₄ + Z₆ - Z₇ + Z₉
────────────────────────────
            6⋅d             

In [57]:
import sympy

sympy.init_printing()

sympy.var("d Z1 Z2 Z3 Z6 Z5 Z4 Z9 Z7 Z8")
p1 = 1/(3*(d)*(d)*(d+(d))*(d**4+(d**4)+(d**4)))

p2 = d**2*(d**4+d**4+d**2*d**2)
p3 = d**2*(d**2)*(d**2-d**2)
p4 = (p2+p3)*(Z1+Z3)

p5 = d**2*(d**4+(d**4)+(d**2*(d**2)))
p6 = d**2*(d**4+(d**4)+(d**2*(d**2)))
p7 = (p5-p6)*(Z4+(Z6))

p9 = d**2*(d**4+(d**4)+(d**2*(d**2)))
p10 = d**2*(d**2)*(d**2-(d**2))
p11 = (p9-p10)*(Z7+(Z9))

p13 = d**2*(d**4*(Z2-3*Z5)+(d**4*(3*Z2-Z5))+(d**4-2*d**2*d**2)*(Z2-(Z5)))

p14 = d**2*(d**4*(Z5-3*Z8)+d**4*(3*Z5-Z8)+(d**4-2*d**2*d**2)*(Z5-Z8))

p15 = 2*(d**2*(d**2)*(d**2-(d**2))*(Z8)+(d**2*(d**2)*(d**2-(d**2))*(Z2)))

q = p1*(p4-(p7)-(p11)+(p13)+(p14)-(p15))#
qk = sympy.simplify(q)
display((qk))

Z₁ + Z₂ + Z₃ - Z₇ - Z₈ - Z₉
───────────────────────────
            6⋅d            

In [2]:
import sympy

sympy.init_printing()

sympy.var("d Z1 Z2 Z3 Z6 Z5 Z4 Z9 Z7 Z8")
p1 = d**2*(Z1+Z3-2*Z2)
p2 = d**2*(Z4+Z6-2*Z5)
p3 = d**2*(Z7+Z9-2*Z8)
p4 = d**4+d**4+d**4
    
r = (p1+p2+p3)/p4
rk = sympy.simplify(r)
display((rk))

Z₁ - 2⋅Z₂ + Z₃ + Z₄ - 2⋅Z₅ + Z₆ + Z₇ - 2⋅Z₈ + Z₉
────────────────────────────────────────────────
                         2                      
                      3⋅d                       

In [12]:
import sympy

sympy.init_printing()

sympy.var("d Z1  Z3 Z6  Z4 Z9 Z7 ")
p1 = d*(d**2*(d+d)+(d**2*d))*(Z3-Z1)
p2 = d*(d**2*d-d**2*d)*(Z4-Z6)
p3 = d*(d**2*(d+d)+(d**2*d))*(Z7-Z9)
p4 = p1-p2+p3

p5 = 2
p6 = d**2*d**2*(d+d)**2
p7 = d**2*(d**2*d**2+d**2*d**2)
p8 = (p6+p7)*p5

s = p4/p8
sk = sympy.simplify(s)
display(sk)

-Z₁ + Z₃ + Z₇ - Z₉
──────────────────
          2       
       4⋅d        

In [33]:
import sympy

sympy.init_printing()

sympy.var("d Z1 Z2 Z3 Z6 Z5 Z4 Z9 Z7 Z8")
p1 = 2/(3*d*d*(d+d)*(d**4+d**4+d**4))

p2 = d*(d**4+d**4+d**2*d**2)
p3 = d**2*d*(d**2-d**2)
p4 = (p2-p3)*(Z1+Z3)

p5 = d*(d**4+d**4+d**2*d**2)
p6 = d*(d**4+d**4+d**2*d**2)
p7 = (p5+p6)*(Z4+Z6)

p9 = d*(d**4+d**4+d**2*d**2)
p10 = d**2*d*(d**2-d**2)
p11 = (p9+p10)*(Z7+Z9)

p13 = d*(d**4*(Z2-(3*Z5))+d**4*((3*Z2)-Z5)+(d**4-(2*d**2*d**2))*(Z2-Z5))

p14 = d*(d**4*((3*Z8)-Z5)+d**4*(Z8-(3*Z5))+(d**4-(2*d**2*d**2))*(Z8-Z5))

p15 = 2*(d**2*d*(d**2-d**2)*(Z8)-(d**2*d*(d**2-d**2)*Z2))

t = p1*(p4-p7+p11+p13+p14-p15)

tk = sympy.simplify(t)
display(tk)

Z₁ + Z₂ + Z₃ - 2⋅Z₄ - 2⋅Z₅ - 2⋅Z₆ + Z₇ + Z₈ + Z₉
────────────────────────────────────────────────
                         2                      
                      3⋅d                       

In [ ]:
p1 = a.pow(2).multiply(c).multiply(d).multiply(d.add(e)).multiply(Z3.subtract(Z1))
p2 = b.multiply(a.pow(2).multiply(d.pow(2)).add(c.pow(2).multiply(e.pow(2)))).multiply(Z6.subtract(Z4))
p3 = a.multiply(c.pow(2)).multiply(e.multiply(d.add(e))).multiply(Z9.subtract(Z7))
p4 = p1.add(p2).add(p3)

p5 = constant2
p6 = a.pow(2).multiply(c.pow(2).multiply(d.add(e).pow(2)))
p7 = b.pow(2).multiply(a.pow(2).multiply(d.pow(2)).add(c.pow(2).multiply(e.pow(2))))
p8 = p6.add(p7).multiply(p5)
    
    p = p4.divide(p8).rename('PDerivative')
p1 = d**2*d*d*(d+d)*(Z3-Z1)
p2 = d*(d**2*d**2+d**2*d**2)*(Z6-Z4)
p3 = d*d**2*d*(d+d)*(Z9-Z7)
p4 = p1+p2+p3

p5 = 2
p6 = d**2*(d**2*(d+(d)**2))
p7 = d**2*(d**2*d**2+d**2*d**2)
p8 = (p6+p7)*p5

p = p4/p8

In [ ]:
p1 = 1/(2*(d)*(d)*(d+(d))*(d**4+(d**4)+(d**4)))

p2 = d**2*(d**4+(d**4)+(d**2*(d**2)))
p3 = d**2*(d**2)*(d**2-(d**2))
p4 = p2+(p3)*(Z1+(Z3))

p5 = d**2*(d**4+(d**4)+(d**2*(d**2)))
p6 = d**2*(d**4+(d**4)+(d**2*(d**2)))
p7 = p5-(p6)*(Z4+(Z6))

p9 = d**2*(d**4+(d**4)+(d**2*(d**2)))
p10 = d**2*(d**2)*(d**2-(d**2))
p11 = p9-(p10)*(Z7+(Z9))

p13 = d**2*(d**4*(Z2-(3*(Z5)))+(d**4*(3*(Z2)-(Z5)))+(d**4-(2*(d**2)*(d**2))*(Z2-(Z5))))

p14 = d**2*(d**4*(Z5-(3*(Z8)))+(d**4*(3*(Z5)-(Z8)))+(d**4-(2*(d**2)*(d**2))*(Z5-(Z8))))

p15 = 2*(d**2*(d**2)*(d**2-(d**2))*(Z8)+(d**2*(d**2)*(d**2-(d**2))*(Z2)))

q = p1*(p4-(p7)-(p11)+(p13)+(p14)-(p15))

p1 = constant1.divide(constant2.multiply(d).multiply(e).multiply(d.add(e)).multiply(a.pow(4).add(b.pow(4)).add(c.pow(4))))

p2 = d.pow(2).multiply(a.pow(4).add(b.pow(4)).add(b.pow(2).multiply(c.pow(2))))
p3 = c.pow(2).multiply(e.pow(2)).multiply(a.pow(2).subtract(b.pow(2)))
p4 = p2.add(p3).multiply(Z1.add(Z3))

p5 = d.pow(2).multiply(a.pow(4).add(c.pow(4)).add(b.pow(2).multiply(c.pow(2))))
p6 = e.pow(2).multiply(a.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
p7 = p5.subtract(p6).multiply(Z4.add(Z6))

p9 = e.pow(2).multiply(b.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
p10 = a.pow(2).multiply(d.pow(2)).multiply(b.pow(2).subtract(c.pow(2)))
p11 = p9.subtract(p10).multiply(Z7.add(Z9))

p13 = d.pow(2).multiply(b.pow(4).multiply(Z2.subtract(constant3.multiply(Z5))).add(c.pow(4).multiply(constant3.multiply(Z2).subtract(Z5))).add(a.pow(4).subtract(constant2.multiply(b.pow(2)).multiply(c.pow(2))).multiply(Z2.subtract(Z5))))

p14 = e.pow(2).multiply(a.pow(4).multiply(Z5.subtract(constant3.multiply(Z8))).add(b.pow(4).multiply(constant3.multiply(Z5).subtract(Z8))).add(c.pow(4).subtract(constant2.multiply(a.pow(2)).multiply(b.pow(2))).multiply(Z5.subtract(Z8))))

p15 = constant2.multiply(a.pow(2).multiply(d.pow(2)).multiply(b.pow(2).subtract(c.pow(2))).multiply(Z8).add(c.pow(2).multiply(e.pow(2)).multiply(a.pow(2).subtract(b.pow(2))).multiply(Z2)))

q = p1.multiply(p4.subtract(p7).subtract(p11).add(p13).add(p14).subtract(p15))

In [ ]:
p1 = d**2*(Z1+Z3-2*Z2)
p2 = d**2*(Z4+Z6-2*Z5)
p3 = d**2*(Z7+Z9-2*Z8)
p4 = d**4+d**4+d**4
    
r = (p1+p2+p3)/p4

p1 = c.pow(2).multiply(Z1.add(Z3).subtract(constant2.multiply(Z2)))
p2 = b.pow(2).multiply(Z4.add(Z6).subtract(constant2.multiply(Z5)))
p3 = a.pow(2).multiply(Z7.add(Z9).subtract(constant2.multiply(Z8)))
p4 = a.pow(4).add(b.pow(4)).add(c.pow(4))

r = p1.add(p2).add(p3).divide(p4)

In [ ]:
p1 = d*(d**2*(d+d)+(d**2*d))*(Z3-Z1)
p2 = d*(d**2*d-d**2*d)*(Z4-Z6)
p3 = d*(d**2*(d+d)+(d**2*d))*(Z7-Z9)
p4 = p1-p2+p3

p5 = 2
p6 = d**2*d**2*(d+d)**2
p7 = d**2*(d**2*d**2+d**2*d**2)
p8 = (p6+p7)*p5

s = p4/p8

p1 = c.multiply(a.pow(2).multiply(d.add(e)).add(b.pow(2).multiply(e))).multiply(Z3.subtract(Z1))
p2 = b.multiply(a.pow(2).multiply(d).subtract(c.pow(2).multiply(e))).multiply(Z4.subtract(Z6))
p3 = a.multiply(c.pow(2).multiply(d.add(e)).add(b.pow(2).multiply(d))).multiply(Z7.subtract(Z9))
p4 = p1.subtract(p2).add(p3)

p5 = constant2
p6 = a.pow(2).multiply(c.pow(2).multiply(d.add(e).pow(2)))
p7 = b.pow(2).multiply(a.pow(2).multiply(d.pow(2)).add(c.pow(2).multiply(e.pow(2))))
p8 = p6.add(p7).multiply(p5)

s = p4.divide(p8)

In [ ]:
p1 = 2/(3*d*d*(d+d)*(d**4+d**4+d**4))

p2 = d*(d**4+d**4+d**2*d**2)
p3 = d**2*d*(d**2-d**2)
p4 = (p2-p3)*(Z1+Z3)

p5 = d*(d**4+d**4+d**2*d**2)
p6 = d*(d**4+d**4+d**2*d**2)
p7 = (p5+p6)*(Z4+Z6)

p9 = d*(d**4+d**4+d**2*d**2)
p10 = d**2*d*(d**2-d**2)
p11 = (p9+p10)*(Z7+Z9)

p13 = d*(d**4*(Z2-(3*Z5))+d**4*((3*Z2)-Z5)+(d**4-(2*d**2*d**2))*(Z2-Z5))

p14 = d*((d**4*((3*Z8)-Z5)+d**4*(Z8-(3*Z5))+(d**4-(2*d**2*d**2)))*(Z8-Z5))

p15 = 2*(d**2*d*(d**2-d**2)*(Z8)-(d**2*d*(d**2-d**2)*Z2))

t = p1*(p4-p7+p11+p13+p14-p15)

p1 = constant2.divide(constant3.multiply(d).multiply(e).multiply(d.add(e)).multiply(a.pow(4).add(b.pow(4)).add(c.pow(4))))

p2 = d.multiply(a.pow(4).add(b.pow(4)).add(b.pow(2).multiply(c.pow(2))))
p3 = c.pow(2).multiply(e).multiply(a.pow(2).subtract(b.pow(2)))
p4 = p2.subtract(p3).multiply(Z1.add(Z3))

p5 = d.multiply(a.pow(4).add(c.pow(4)).add(b.pow(2).multiply(c.pow(2))))
p6 = e.multiply(a.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
p7 = p5.add(p6).multiply(Z4.add(Z6))

p9 = e.multiply(b.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
p10 = a.pow(2).multiply(d).multiply(b.pow(2).subtract(c.pow(2)))
p11 = p9.add(p10).multiply(Z7.add(Z9))

p13 = d.multiply(b.pow(4).multiply(Z2.subtract(constant3.multiply(Z5))).add(c.pow(4).multiply(constant3.multiply(Z2).subtract(Z5))).add(a.pow(4).subtract(constant2.multiply(b.pow(2)).multiply(c.pow(2))).multiply(Z2.subtract(Z5))))

p14 = e.multiply(a.pow(4).multiply(constant3.multiply(Z8).subtract(Z5)).add(b.pow(4).multiply(Z8.subtract(constant3.multiply(Z5)))).add(c.pow(4).subtract(constant2.multiply(a.pow(2)).multiply(b.pow(2))).multiply(Z8.subtract(Z5))))

p15 = constant2.multiply(a.pow(2).multiply(d).multiply(b.pow(2).subtract(c.pow(2))).multiply(Z8).subtract(c.pow(2).multiply(e).multiply(a.pow(2).subtract(b.pow(2))).multiply(Z2)))

t = p1.multiply(p4.subtract(p7).add(p11).add(p13).add(p14).subtract(p15))

In [ ]:
import ee
import math

# Functions for calculating parameters

def calculateParameters(dem):

  # Defining kernels to retrieve 3x3 window elevations

  # Weights for a 3x3 kernel
  w00 = [0, 0, 0]
  w11 = [1, 0, 0]
  w12 = [0, 1, 0]
  w13 = [0, 0, 1]

  # Neighborhood indices
  n1 = [w11, w00, w00]
  n2 = [w12, w00, w00]
  n3 = [w13, w00, w00]
  n4 = [w00, w11, w00]
  n5 = [w00, w12, w00]
  n6 = [w00, w13, w00]
  n7 = [w00, w00, w11]
  n8 = [w00, w00, w12]
  n9 = [w00, w00, w13]

  # Kernel for each neighborhood index
  kerneln1 = ee.Kernel.fixed(3, 3, n1, 1, 1, False)
  kerneln2 = ee.Kernel.fixed(3, 3, n2, 1, 1, False)
  kerneln3 = ee.Kernel.fixed(3, 3, n3, 1, 1, False)
  kerneln4 = ee.Kernel.fixed(3, 3, n4, 1, 1, False)
  kerneln5 = ee.Kernel.fixed(3, 3, n5, 1, 1, False)
  kerneln6 = ee.Kernel.fixed(3, 3, n6, 1, 1, False)
  kerneln7 = ee.Kernel.fixed(3, 3, n7, 1, 1, False)
  kerneln8 = ee.Kernel.fixed(3, 3, n8, 1, 1, False)
  kerneln9 = ee.Kernel.fixed(3, 3, n9, 1, 1, False)

  # Function to compute single neighborhood values
  def addNParameters(dem):
    N1 = dem.convolve(kerneln1).rename('N1')
    N2 = dem.convolve(kerneln2).rename('N2')
    N3 = dem.convolve(kerneln3).rename('N3')
    N4 = dem.convolve(kerneln4).rename('N4')
    N5 = dem.convolve(kerneln5).rename('N5')
    N6 = dem.convolve(kerneln6).rename('N6')
    N7 = dem.convolve(kerneln7).rename('N7')
    N8 = dem.convolve(kerneln8).rename('N8')
    N9 = dem.convolve(kerneln9).rename('N9')
    return dem.addBands([N1,N2,N3,N4,N5,N6,N7,N8,N9])

  # Elevation values for each neighborhood
  demZPar = addNParameters(dem)
  demZParameters = demZPar.rename(['Elevation','Z1','Z2','Z3','Z4','Z5','Z6','Z7','Z8','Z9'])
  #print(demZParameters, 'DEM with Z parameters')

  # Defining positions
  demPositions = dem.addBands(ee.Image.pixelLonLat())

  # Longitude
  long = demPositions.select('longitude')
  longNPar = addNParameters(long)
  longNParameters = longNPar.rename(['longitude','longN1','longN2','longN3','longN4','longN5','longN6','longN7','longN8','longN9'])
  #print(longNParameters, 'Longitude with neighborhood parameters')

  # Latitude
  lat = demPositions.select('latitude')
  latNPar = addNParameters(lat)
  latNParameters = latNPar.rename(['latitude','latN1','latN2','latN3','latN4','latN5','latN6','latN7','latN8','latN9'])
  #print(latNParameters, 'Latitude with neighborhood parameters')

  # Function of haversine formula to retrieve distances between two neighborhood points on a spheroidal grid
  def haversineFunction(demLat, demLong, latNeigh1, latNeigh2, longNeigh1, longNeigh2):
    phi1 = demLat.select(ee.String(latNeigh1)).divide(180).multiply(math.pi) # to Radians
    phi2 = demLat.select(ee.String(latNeigh2)).divide(180).multiply(math.pi) # to Radians
    lambda1 = demLong.select(ee.String(longNeigh1)).divide(180).multiply(math.pi) # to Radians
    lambda2 = demLong.select(ee.String(longNeigh2)).divide(180).multiply(math.pi) # to Radians
    
    deltaphi = phi2.subtract(phi1) # (phi2 - phi1)
    deltalambda = lambda2.subtract(lambda1) # (lambda2 - lambda1)
    p1 = deltaphi.divide(2).sin().multiply(deltaphi.divide(2).sin()) # sin(deltaphi/2) * sin(deltaphi/2)
    p2 = phi1.cos().multiply(phi2.cos()) # cos(phi1) * cos(phi2)
    p3 = deltalambda.divide(2).sin().multiply(deltalambda.divide(2).sin()) # sin(deltalambda/2) * sin(deltalambda/2)
    j = p2.multiply(p3).add(p1) # j = p1 + p2 * p3
    p4 = ee.Image(ee.Number(1)) # p4 = image with constant 1
    p5 = ee.Image(ee.Number(2)) # p5 = image with constant 2
    p6 = p4.subtract(j).sqrt() # sqrt(1-j)
    p7 = j.sqrt() # sqrt(a)
    p8 = p6.atan2(p7) # atan2(p6,p7)
    k = p5.multiply(p8) # k = 2 * p8
    R = ee.Image(ee.Number(6371000)) # approximate radius of Earth
    l = R.multiply(k) # l = R * k which is the distance between two points
    
    return l

  # Distance values
  lenghtOfE = haversineFunction(latNParameters, longNParameters, 'latN1', 'latN4', 'longN1', 'longN4').rename('e')
  lenghtOfD = haversineFunction(latNParameters, longNParameters, 'latN4', 'latN7', 'longN4', 'longN7').rename('d')
  lenghtOfC = haversineFunction(latNParameters, longNParameters, 'latN1', 'latN2', 'longN1', 'longN2').rename('c')
  lenghtOfB = haversineFunction(latNParameters, longNParameters, 'latN4', 'latN5', 'longN4', 'longN5').rename('b')
  lenghtOfA = haversineFunction(latNParameters, longNParameters, 'latN7', 'latN8', 'longN7', 'longN8').rename('a')

  # Merging all the parameters in a single image
  demCalculations = (demZParameters.addBands(lenghtOfA)
                                      .addBands(lenghtOfB)
                                      .addBands(lenghtOfC)
                                      .addBands(lenghtOfD)
                                      .addBands(lenghtOfE))
  return demCalculations

# Functions for calculating terrain derivatives  //

def calculateDerivatives(parameters):

  # Functions for Derivatives and Terrain Attributes

  def addPDerivative(parameters):
    a = parameters.select('a')
    b = parameters.select('b')
    c = parameters.select('c')
    d = parameters.select('d')
    e = parameters.select('e')
    Z1 = parameters.select('Z1')
    Z3 = parameters.select('Z3')
    Z4 = parameters.select('Z4')
    Z6 = parameters.select('Z6')
    Z7 = parameters.select('Z7')
    Z9 = parameters.select('Z9')
    constant2 = ee.Image(ee.Number(2))
    
    p1 = a.pow(2).multiply(c).multiply(d).multiply(d.add(e)).multiply(Z3.subtract(Z1))
    p2 = b.multiply(a.pow(2).multiply(d.pow(2)).add(c.pow(2).multiply(e.pow(2)))).multiply(Z6.subtract(Z4))
    p3 = a.multiply(c.pow(2)).multiply(e.multiply(d.add(e))).multiply(Z9.subtract(Z7))
    p4 = p1.add(p2).add(p3)
    
    p5 = constant2
    p6 = a.pow(2).multiply(c.pow(2).multiply(d.add(e).pow(2)))
    p7 = b.pow(2).multiply(a.pow(2).multiply(d.pow(2)).add(c.pow(2).multiply(e.pow(2))))
    p8 = p6.add(p7).multiply(p5)
    
    p = p4.divide(p8).rename('PDerivative')
    
    return p
  

  def addQDerivative(parameters):
    a = parameters.select('a')
    b = parameters.select('b')
    c = parameters.select('c')
    d = parameters.select('d')
    e = parameters.select('e')
    Z1 = parameters.select('Z1')
    Z2 = parameters.select('Z2')
    Z3 = parameters.select('Z3')
    Z4 = parameters.select('Z4')
    Z5 = parameters.select('Z5')
    Z6 = parameters.select('Z6')
    Z7 = parameters.select('Z7')
    Z8 = parameters.select('Z8')
    Z9 = parameters.select('Z9')
    constant1 = ee.Image(ee.Number(1))
    constant2 = ee.Image(ee.Number(2))
    constant3 = ee.Image(ee.Number(3))
    
    p1 = constant1.divide(constant2.multiply(d).multiply(e).multiply(d.add(e)).multiply(a.pow(4).add(b.pow(4)).add(c.pow(4))))
    
    p2 = d.pow(2).multiply(a.pow(4).add(b.pow(4)).add(b.pow(2).multiply(c.pow(2))))
    p3 = c.pow(2).multiply(e.pow(2)).multiply(a.pow(2).subtract(b.pow(2)))
    p4 = p2.add(p3).multiply(Z1.add(Z3))
    
    p5 = d.pow(2).multiply(a.pow(4).add(c.pow(4)).add(b.pow(2).multiply(c.pow(2))))
    p6 = e.pow(2).multiply(a.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
    p7 = p5.subtract(p6).multiply(Z4.add(Z6))
    
    p9 = e.pow(2).multiply(b.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
    p10 = a.pow(2).multiply(d.pow(2)).multiply(b.pow(2).subtract(c.pow(2)))
    p11 = p9.subtract(p10).multiply(Z7.add(Z9))
    
    p13 = d.pow(2).multiply(b.pow(4).multiply(Z2.subtract(constant3.multiply(Z5))).add(c.pow(4).multiply(constant3.multiply(Z2).subtract(Z5))).add(a.pow(4).subtract(constant2.multiply(b.pow(2)).multiply(c.pow(2))).multiply(Z2.subtract(Z5))))
    
    p14 = e.pow(2).multiply(a.pow(4).multiply(Z5.subtract(constant3.multiply(Z8))).add(b.pow(4).multiply(constant3.multiply(Z5).subtract(Z8))).add(c.pow(4).subtract(constant2.multiply(a.pow(2)).multiply(b.pow(2))).multiply(Z5.subtract(Z8))))
    
    p15 = constant2.multiply(a.pow(2).multiply(d.pow(2)).multiply(b.pow(2).subtract(c.pow(2))).multiply(Z8).add(c.pow(2).multiply(e.pow(2)).multiply(a.pow(2).subtract(b.pow(2))).multiply(Z2)))
    
    q = p1.multiply(p4.subtract(p7).subtract(p11).add(p13).add(p14).subtract(p15)).rename('QDerivative')
    
    return q

  def addRDerivative(parameters):
    a = parameters.select('a')
    b = parameters.select('b')
    c = parameters.select('c')
    Z1 = parameters.select('Z1')
    Z2 = parameters.select('Z2')
    Z3 = parameters.select('Z3')
    Z4 = parameters.select('Z4')
    Z5 = parameters.select('Z5')
    Z6 = parameters.select('Z6')
    Z7 = parameters.select('Z7')
    Z8 = parameters.select('Z8')
    Z9 = parameters.select('Z9')
    constant2 = ee.Image(ee.Number(2))
    
    p1 = c.pow(2).multiply(Z1.add(Z3).subtract(constant2.multiply(Z2)))
    p2 = b.pow(2).multiply(Z4.add(Z6).subtract(constant2.multiply(Z5)))
    p3 = a.pow(2).multiply(Z7.add(Z9).subtract(constant2.multiply(Z8)))
    p4 = a.pow(4).add(b.pow(4)).add(c.pow(4))
    
    r = p1.add(p2).add(p3).divide(p4).rename('RDerivative')
    
    return r

  def addSDerivative(parameters):
    a = parameters.select('a')
    b = parameters.select('b')
    c = parameters.select('c')
    d = parameters.select('d')
    e = parameters.select('e')
    Z1 = parameters.select('Z1')
    Z2 = parameters.select('Z2')
    Z3 = parameters.select('Z3')
    Z4 = parameters.select('Z4')
    Z5 = parameters.select('Z5')
    Z6 = parameters.select('Z6')
    Z7 = parameters.select('Z7')
    Z8 = parameters.select('Z8')
    Z9 = parameters.select('Z9')
    constant2 = ee.Image(ee.Number(2))
    
    p1 = c.multiply(a.pow(2).multiply(d.add(e)).add(b.pow(2).multiply(e))).multiply(Z3.subtract(Z1))
    p2 = b.multiply(a.pow(2).multiply(d).subtract(c.pow(2).multiply(e))).multiply(Z4.subtract(Z6))
    p3 = a.multiply(c.pow(2).multiply(d.add(e)).add(b.pow(2).multiply(d))).multiply(Z7.subtract(Z9))
    p4 = p1.subtract(p2).add(p3)
    
    p5 = constant2
    p6 = a.pow(2).multiply(c.pow(2).multiply(d.add(e).pow(2)))
    p7 = b.pow(2).multiply(a.pow(2).multiply(d.pow(2)).add(c.pow(2).multiply(e.pow(2))))
    p8 = p6.add(p7).multiply(p5)
    
    s = p4.divide(p8).rename('SDerivative')
    
    return s

  def addTDerivative(parameters):
    a = parameters.select('a')
    b = parameters.select('b')
    c = parameters.select('c')
    d = parameters.select('d')
    e = parameters.select('e')
    Z1 = parameters.select('Z1')
    Z2 = parameters.select('Z2')
    Z3 = parameters.select('Z3')
    Z4 = parameters.select('Z4')
    Z5 = parameters.select('Z5')
    Z6 = parameters.select('Z6')
    Z7 = parameters.select('Z7')
    Z8 = parameters.select('Z8')
    Z9 = parameters.select('Z9')
    constant1 = ee.Image(ee.Number(1))
    constant2 = ee.Image(ee.Number(2))
    constant3 = ee.Image(ee.Number(3))
    
    p1 = constant2.divide(constant3.multiply(d).multiply(e).multiply(d.add(e)).multiply(a.pow(4).add(b.pow(4)).add(c.pow(4))))
    
    p2 = d.multiply(a.pow(4).add(b.pow(4)).add(b.pow(2).multiply(c.pow(2))))
    p3 = c.pow(2).multiply(e).multiply(a.pow(2).subtract(b.pow(2)))
    p4 = p2.subtract(p3).multiply(Z1.add(Z3))
    
    p5 = d.multiply(a.pow(4).add(c.pow(4)).add(b.pow(2).multiply(c.pow(2))))
    p6 = e.multiply(a.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
    p7 = p5.add(p6).multiply(Z4.add(Z6))
    
    p9 = e.multiply(b.pow(4).add(c.pow(4)).add(a.pow(2).multiply(b.pow(2))))
    p10 = a.pow(2).multiply(d).multiply(b.pow(2).subtract(c.pow(2)))
    p11 = p9.add(p10).multiply(Z7.add(Z9))
    
    p13 = d.multiply(b.pow(4).multiply(Z2.subtract(constant3.multiply(Z5))).add(c.pow(4).multiply(constant3.multiply(Z2).subtract(Z5))).add(a.pow(4).subtract(constant2.multiply(b.pow(2)).multiply(c.pow(2))).multiply(Z2.subtract(Z5))))
    
    p14 = e.multiply(a.pow(4).multiply(constant3.multiply(Z8).subtract(Z5)).add(b.pow(4).multiply(Z8.subtract(constant3.multiply(Z5)))).add(c.pow(4).subtract(constant2.multiply(a.pow(2)).multiply(b.pow(2))).multiply(Z8.subtract(Z5))))
    
    p15 = constant2.multiply(a.pow(2).multiply(d).multiply(b.pow(2).subtract(c.pow(2))).multiply(Z8).subtract(c.pow(2).multiply(e).multiply(a.pow(2).subtract(b.pow(2))).multiply(Z2)))
    
    t = p1.multiply(p4.subtract(p7).add(p11).add(p13).add(p14).subtract(p15)).rename('TDerivative')
    
    return t

  def signPFunction(pDerivative):
    signP = pDerivative.expression("(b('PDerivative') > 0) ? 1" + ": (b('PDerivative') == 0) ? 0" + ": -1").rename("signP")
    
    return signP

  def signQFunction(qDerivative):
    signQ = qDerivative.expression("(b('QDerivative') > 0) ? 1" + ": (b('QDerivative') == 0) ? 0" + ": -1").rename("signQ")
    
    return signQ

  # Calculating the derivatives

  pDerivative = addPDerivative(parameters)
  qDerivative = addQDerivative(parameters)
  rDerivative = addRDerivative(parameters)
  sDerivative = addSDerivative(parameters)
  tDerivative = addTDerivative(parameters)
  signP = signPFunction(pDerivative)
  signQ = signQFunction(qDerivative)

  demWithDerivatives = (parameters.addBands(pDerivative)
                                    .addBands(qDerivative)
                                    .addBands(rDerivative)
                                    .addBands(sDerivative)
                                    .addBands(tDerivative)
                                    .addBands(signP)
                                    .addBands(signQ))

  return demWithDerivatives

# Functions for calculating terrain attributes

def calculateAttributes(derivatives):

  def slopeFunction(derivatives):
    p = derivatives.select('PDerivative')
    q = derivatives.select('QDerivative')
    
    p2 = p.pow(2).rename('A')
    q2 = q.pow(2).rename('A')
    p2q2 = ee.ImageCollection([p2,q2])
    sumP2q2 = p2q2.sum()
    sqrtSumP2q2 = sumP2q2.sqrt()
    slope = sqrtSumP2q2.atan().multiply(180).divide(math.pi).rename('Slope')
    
    return slope

  def aspectFunction(derivatives):
    p = derivatives.select('PDerivative')
    q = derivatives.select('QDerivative')
    signP = derivatives.select('signP')
    signQ = derivatives.select('signQ')
    constant1 = ee.Image(ee.Number(1))
    constantNeg1 = ee.Image(ee.Number(-1))
    constant90 = ee.Image(ee.Number(90))
    constant180 = ee.Image(ee.Number(180))
    
    p1 = constantNeg1.multiply(constant90).multiply(constant1.subtract(signQ)).multiply(constant1.subtract(signP.abs()))
    p2 = constant180.multiply(constant1.add(signP))
    p3 = constant180.divide(math.pi).multiply(signP)
    p4 = constantNeg1.multiply(q).divide(p.pow(2).add(q.pow(2)).sqrt()).acos()
    A = p1.add(p2).subtract(p3.multiply(p4)).rename('Aspect')
    
    return A

  def hillshadeFunction(aspect):
    p = derivatives.select('PDerivative')
    q = derivatives.select('QDerivative')
    theta = ee.Image(ee.Number(45)) # Azimuth
    psi = ee.Image(ee.Number(315)) # Elevation angle
    constant1 = ee.Image(ee.Number(1))
    
    p1 = constant1.subtract(p.multiply(theta.sin()).multiply(constant1.divide(psi))).subtract(q.multiply(theta.cos()).multiply(constant1.divide(psi)))
    p2 = constant1.add(p.pow(2)).add(q.pow(2)).sqrt().multiply(constant1.add(theta.sin().multiply(constant1.divide(psi)).pow(2)).add(theta.cos().multiply(constant1.divide(psi)).pow(2)).sqrt())
    AH = p1.divide(p2).rename('Hillshade')
    
    return AH


  def northernnessFunction(aspect):
    A = aspect.select('Aspect')
    
    AN = A.multiply(math.pi).divide(180).cos().rename('Northness')
    
    return AN

  def easternnessFunction(aspect):
    A = aspect.select('Aspect')
    
    AE = A.multiply(math.pi).divide(180).sin().rename('Eastness')
    
    return AE

  def horizontalCurvatureFunction(derivatives):
    p = derivatives.select('PDerivative')
    q = derivatives.select('QDerivative')
    r = derivatives.select('RDerivative')
    s = derivatives.select('SDerivative')
    t = derivatives.select('TDerivative')
    constantNeg1 = ee.Image(ee.Number(-1))
    constant1 = ee.Image(ee.Number(1))
    constant2 = ee.Image(ee.Number(2))
    
    p1 = q.pow(2).multiply(r).subtract(constant2.multiply(p).multiply(q).multiply(s)).add(p.pow(2).multiply(t))
    p2 = p.pow(2).add(q.pow(2)).multiply(constant1.add(p.pow(2)).add(q.pow(2)).sqrt())
    kh = constantNeg1.multiply(p1.divide(p2)).rename('HorizontalCurvature')
    
    return kh

  def verticalCurvatureFunction(derivatives):
    p = derivatives.select('PDerivative')
    q = derivatives.select('QDerivative')
    r = derivatives.select('RDerivative')
    s = derivatives.select('SDerivative')
    t = derivatives.select('TDerivative')
    constantNeg1 = ee.Image(ee.Number(-1))
    constant1 = ee.Image(ee.Number(1))
    constant2 = ee.Image(ee.Number(2))

    p2 = p.pow(2).multiply(r).add(constant2.multiply(p).multiply(q).multiply(s)).add(q.pow(2).multiply(t))
    p3 = p.pow(2).add(q.pow(2)).multiply(constant1.add(p.pow(2)).add(q.pow(2)).pow(3).sqrt())
    kv = constantNeg1.multiply(p2.divide(p3)).rename('VerticalCurvature')
    
    return kv

  def meanCurvatureFunction(derivatives):
    p = derivatives.select('PDerivative')
    q = derivatives.select('QDerivative')
    r = derivatives.select('RDerivative')
    s = derivatives.select('SDerivative')
    t = derivatives.select('TDerivative')
    constantNeg1 = ee.Image(ee.Number(-1))
    constant1 = ee.Image(ee.Number(1))
    constant2 = ee.Image(ee.Number(2))

    p2 = constant1.add(q.pow(2)).multiply(r).subtract(constant2.multiply(p).multiply(q).multiply(s)).add(constant1.add(p.pow(2)).multiply(t))
    p3 = constant2.multiply(constant1.add(p.pow(2)).add(q.pow(2)).pow(3).sqrt())
    km = constantNeg1.multiply(p2.divide(p3)).rename('MeanCurvature')
    
    return km

  def gaussianCurvatureFunction(derivatives):
    p = derivatives.select('PDerivative')
    q = derivatives.select('QDerivative')
    r = derivatives.select('RDerivative')
    s = derivatives.select('SDerivative')
    t = derivatives.select('TDerivative')
    constant1 = ee.Image(ee.Number(1))
    
    p1 = r.multiply(t).subtract(s.pow(2))
    p2 = constant1.add(p.pow(2)).add(p.pow(2)).pow(2)
    kg = p1.divide(p2).rename('GaussianCurvature')
    
    return kg

  def minimalCurvatureFunction(gaussian, mean):
    K = gaussian.select('GaussianCurvature')
    H = mean.select('MeanCurvature')
    kmin = H.subtract(H.pow(2).subtract(K).sqrt()).rename('MinimalCurvature')
    
    return kmin

  def maximalCurvatureFunction(gaussian, mean):
    K = gaussian.select('GaussianCurvature')
    H = mean.select('MeanCurvature')
    kmax = H.add(H.pow(2).subtract(K).sqrt()).rename('MaximalCurvature')
    return kmax

  def shapeIndexFunction(gaussian, mean):
    K = gaussian.select('GaussianCurvature').rename('K')
    H = mean.select('MeanCurvature').rename('H')
    constant2 = ee.Image(ee.Number(2))
    
    index = constant2.divide(math.pi).multiply(H.divide(H.pow(2).subtract(K).sqrt())).rename('ShapeIndex')
    
    return index

  # Calculating the Attributes

  slope = slopeFunction(derivatives)
  aspect = aspectFunction(derivatives)
  hillshade = hillshadeFunction(derivatives)
  northernness = northernnessFunction(aspect)
  easternness = easternnessFunction(aspect)
  horizontalCurvature = horizontalCurvatureFunction(derivatives)
  verticalCurvature = verticalCurvatureFunction(derivatives)
  meanCurvature = meanCurvatureFunction(derivatives)
  gaussianCurvature = gaussianCurvatureFunction(derivatives)
  minimalCurvature = minimalCurvatureFunction(gaussianCurvature, meanCurvature)
  maximalCurvature = maximalCurvatureFunction(gaussianCurvature, meanCurvature)
  shapeIndex = shapeIndexFunction(gaussianCurvature, meanCurvature)

  demWithAttributes = (derivatives.addBands(slope)
                                    .addBands(aspect)
                                    .addBands(hillshade)
                                    .addBands(northernness)
                                    .addBands(easternness)
                                    .addBands(horizontalCurvature)
                                    .addBands(verticalCurvature)
                                    .addBands(meanCurvature)
                                    .addBands(gaussianCurvature)
                                    .addBands(minimalCurvature)
                                    .addBands(maximalCurvature)
                                    .addBands(shapeIndex))

  return (demWithAttributes.select('Elevation', 'Slope', 'Aspect', 'Hillshade', 'Northness', 'Eastness',
                                  'HorizontalCurvature', 'VerticalCurvature', 'MeanCurvature',
                                  'GaussianCurvature', 'MinimalCurvature', 'MaximalCurvature', 'ShapeIndex'))


def terrainAnalysis(dem: ee.Image, bbox: ee.Geometry | None = None) -> ee.Image:
  """
  Calculate all terrain attributes for a given DEM and region.

    Parameters:
      dem (ee.Image): 
        An image representing elevation values.
      bbox (ee.Geometry | None):
        A geometry over which terrain attributes 
        will be calculated.

    Returns:
      attributes (ee.Image): 
        An image with calculated terrain attributes
        with the following bands: Elevation, Slope, Aspect, Hillshade,
        Northness, Eastness, HorizontalCurvature, VerticalCurvature,
        MeanCurvature, GaussianCurvature, MinimalCurvature, MaximalCurvature
  """
  parameters = calculateParameters(dem)
  derivatives = calculateDerivatives(parameters)
  attributes = calculateAttributes(derivatives)
  if bbox is not None:
    return attributes.clip(bbox)
  return(attributes)

# Additional features

def makeVisualization(result: ee.Image, bandName: str, 
                      zoomLevel: str, bbox: ee.Geometry, palette: str) -> ee.Image:
  """
  Generate a visualization for a given band at a particular zoom level.
  This function will dynamically determine the appropriate color scale
  to display the image from a choice of palettes.

  Parameters:
    result (ee.Image): 
      Image to be visualized.
    bandName (str): 
      Band within result to be visualized.
    zoomLevel (str): 
      Desired zoom level. Must be of form "levelX",
      where X is from 0-15, inclusive.
    bbox (ee.Geometry):
      Geometry for which the color scale should be calculated. The 5th and 95th
      percentile of result[bandName] are calculated in this region.
    palette (str):
      Palette to use for the visualization. Must be on of: "rainbow", "inferno",
      "cubehelix", "red2green", "green2red", "elevation", "aspect".

  Returns:
    visualization (ee.Image):
      An image with red, green, and blue bands set according to the
      visualization.

  """
  
  levelsDic = ee.Dictionary({
        'level0': {'zoom': 0, 'scale': 157000},
        'level1': {'zoom': 1, 'scale': 78000},
        'level2': {'zoom': 2, 'scale': 39000},
        'level3': {'zoom': 3, 'scale': 20000},
        'level4': {'zoom': 4, 'scale': 10000},
        'level5': {'zoom': 5, 'scale': 5000},
        'level6': {'zoom': 6, 'scale': 2000},
        'level7': {'zoom': 7, 'scale': 1000},
        'level8': {'zoom': 8, 'scale': 611},
        'level9': {'zoom': 9, 'scale': 306},
        'level10': {'zoom': 10, 'scale': 153},
        'level11': {'zoom': 11, 'scale': 76},
        'level12': {'zoom': 12, 'scale': 38},
        'level13': {'zoom': 13, 'scale': 19},
        'level14': {'zoom': 14, 'scale': 10},
        'level15': {'zoom': 15, 'scale': 5},
  })

  levelSelected = ee.Dictionary(levelsDic.get(zoomLevel))

  imageSelected = result.select(bandName).rename('selection')

  minMaxLegend = imageSelected.reduceRegion(
          reducer=ee.Reducer.percentile(percentiles=[5,95], outputNames=['perc5','perc95']),
          geometry=bbox,
          scale=levelSelected.get('scale'),
          bestEffort=True)

  palettes = ee.Dictionary({
    'rainbow': '6e40aa, be3caf, fe4b83, ff7747, e3b62f, b0ef5a, 53f666, 1edfa2, 23acd8, 4c6fdc',
    'inferno': '000004, 160b39, 420a68, 6a176e, 932667, ba3655, dd513a, f3761b, fca50a, f6d746',
    'cubehelix': '163d4e, 1f6642, 53792f, a07949, d07e93, d09cd9, c1caf3',
    'red2green': 'a50026, d3322b, f16d43, fcab63, fedc8c, f9f7ae, d7ee8e, a4d86f, 64bc61, 23964f',
    'green2red': '23964f, 64bc61, a4d86f, d7ee8e, f9f7ae, fedc8c, fcab63, f16d43, d3322b, a50026',
    'elevation': 'b0f3be, e0fbb2, b8de76, 27a52a, 34883c, 9ca429, f8b004, c04a02, c04a02, 870800, 741805, 6c2a0a, 7d4a2b, 9c8170, b5b5b5, dad8da',
    'aspect': 'red, green, blue, yellow, red',    'hillshade': 'black, white'
  })
  
  visualization = imageSelected.visualize({
    min: minMaxLegend.get('selection_perc5'),
    max: minMaxLegend.get('selection_perc95'),
    palette: palettes.get(palette)
  })
  
  return visualization

def logTransformation(result: ee.Image, bandName: str) -> ee.Image:
  """
  Apply a log10 transformation to an image.

  Parameters:
    result (ee.Image):
      Image to be transformed.
    bandName (str):
      Band in image to be transformed.
  
  Returns:
    logValue (ee.Image):
      Log-transformed image.
  """
  selection = result.select(bandName).rename('selection')
  sign = selection.expression("(b('selection') > 0) ? 1" + ": (b('selection') == 0) ? 0" + ": -1").rename("sign")
  constant1 = ee.Image(ee.Number(1))
  constant10 = ee.Image(ee.Number(10))
  logValues = selection.abs().multiply(constant10.pow(4)).add(1).log10().multiply(sign).rename(bandName)
  
  return logValues